# Canada Critical Mineral Priority Mapper 

A clean, self-contained notebook that replicates the web app functionality locally.

**Features:**
- Mimics the exact web app interface and behavior
- Downloads data from public S3 bucket automatically
- Runs all processing locally (no server required)
- Produces identical results to the web app
- Works locally in VS Code, Jupyter Notebook, or online in Google Colab

**Instructions:**
1. Run the setup cells (packages & configuration)
2. Use the interface to select mineral type and priorities
3. Optionally upload custom GeoTIFF files
4. Click "Upload & Process" to run the analysis
5. View results and download files

In [ ]:
# 📦 PACKAGE INSTALLATION - Run this cell first
import subprocess
import sys
import os

def install_package(package):
    """Install a package if it's not already installed"""
    try:
        __import__(package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
        print(f"✅ {package} installed successfully")

# Essential packages for the analysis
packages = [
    'ipywidgets', 'geopandas', 'rasterio', 'matplotlib',
    'scipy', 'pyproj', 'shapely', 'requests', 'plotly', 'ipyfilechooser', 'py7zr'
]

print("🚀 Installing required packages...")
for package in packages:
    install_package(package)

print("\n🎉 All packages ready!")

In [ ]:
# IMPROVED ENVIRONMENT DETECTION - Run this cell first
import sys
import warnings
import os
warnings.filterwarnings('ignore')

# Detect environment with improved logic
IN_COLAB = False
IS_LOCAL = False
CLOUD_ENV_NAME = None

# 1. Check for Google Colab (most specific)
try:
    import google.colab
    # Verify it's actually Colab (not just imported in Kaggle)
    if 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython()):
        IN_COLAB = True
        CLOUD_ENV_NAME = "Google Colab"
        print("🔍 Google Colab environment detected")
        
        # Colab-specific setup
        from google.colab import files, drive
        
        # Mount Google Drive for persistent storage (optional)
        try:
            drive.mount('/content/drive')
            WORK_DIR = '/content/drive/MyDrive/ESG_Mapper_Results'
            print("✅ Google Drive mounted - using persistent storage")
        except:
            WORK_DIR = '/content/esg_results'
            print("📁 Using local Colab storage")
except:
    pass

# 2. If not Colab, check for other cloud notebook environments
if not IN_COLAB:
    # Kaggle
    if os.path.exists('/kaggle/working') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        CLOUD_ENV_NAME = "Kaggle"
        WORK_DIR = '/kaggle/working/esg_results'
        print("🔍 Kaggle environment detected")
    
    # AWS SageMaker
    elif '/opt/ml' in os.getcwd() or 'SageMaker' in os.environ.get('AWS_EXECUTION_ENV', ''):
        CLOUD_ENV_NAME = "AWS SageMaker"
        WORK_DIR = '/home/ec2-user/SageMaker/esg_results'
        print("🔍 AWS SageMaker environment detected")
    
    # Azure ML
    elif 'AZUREML_RUN_ID' in os.environ:
        CLOUD_ENV_NAME = "Azure ML"
        WORK_DIR = './outputs/esg_results'
        print("🔍 Azure ML environment detected")
    
    # Paperspace Gradient
    elif 'PAPERSPACE_NOTEBOOK_REPO_ID' in os.environ or 'PS_API_KEY' in os.environ:
        CLOUD_ENV_NAME = "Paperspace Gradient"
        WORK_DIR = '/notebooks/esg_results'
        print("🔍 Paperspace Gradient environment detected")
    
    # Databricks
    elif 'DATABRICKS_RUNTIME_VERSION' in os.environ:
        CLOUD_ENV_NAME = "Databricks"
        WORK_DIR = '/dbfs/esg_results'
        print("🔍 Databricks environment detected")
    
    # Generic Jupyter/JupyterLab (local)
    else:
        IS_LOCAL = True
        WORK_DIR = './esg_results'
        print("💻 Local Jupyter environment detected")

# Create working directory
os.makedirs(WORK_DIR, exist_ok=True)
print(f"📂 Working directory: {WORK_DIR}")

# Summary
if IN_COLAB:
    print("🌐 Environment: Google Colab (direct hosting supported)")
elif IS_LOCAL:
    print("💻 Environment: Local Jupyter (browser access available)")
else:
    print(f"☁️ Environment: {CLOUD_ENV_NAME} (download and host locally required)")

In [ ]:
# 📚 IMPORT LIBRARIES
import json
import time
import tempfile
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from rasterio.warp import reproject
from pyproj import Transformer
from shapely.geometry import Point
from scipy.stats import rankdata
import ipywidgets as widgets
from IPython.display import display, HTML, Image, clear_output
from pathlib import Path
import hashlib
from ipyfilechooser import FileChooser



# GitHub Setup
import requests
import py7zr

GITHUB_REPO = "NRCan/CCMPM"
GITHUB_BRANCH = "main"
GITHUB_DATA_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}/data/"

print("✅ All libraries imported successfully")
print(f"✅ GitHub data source: {GITHUB_DATA_URL}")

In [ ]:
# 🔧 EMBEDDED OPTIMIZATION LIBRARY (from optimization_lib.py)
class MiningSiteOptimization:
    def __init__(self, raster_paths_maximize, raster_paths_minimize):
        self.rasters = []
        self.profiles = []
        self.original_shape = None
        self.raster_paths_maximize = raster_paths_maximize
        self.raster_paths_minimize = raster_paths_minimize
        self.rasters_maximize = self.load_rasters(raster_paths_maximize)
        self.rasters_minimize = self.load_rasters(raster_paths_minimize, invert=True)
        self.rasters = self.rasters_maximize + self.rasters_minimize
        self.valid_indices = ~np.isnan(self.rasters[0].flatten())
        for raster in self.rasters:
            self.valid_indices &= ~np.isnan(raster.flatten())
        self.prepare_objective_array()
        
        # Create display names and flags
        self.raster_display_names = []
        self.minimize_flags = []
        for path in raster_paths_maximize:
            name = os.path.splitext(os.path.basename(path))[0]
            self.raster_display_names.append(name)
            self.minimize_flags.append(False)
        for path in raster_paths_minimize:
            name = os.path.splitext(os.path.basename(path))[0]
            self.raster_display_names.append(name)
            self.minimize_flags.append(True)
    
    @property
    def num_valid_points(self):
        return self.objective_values.shape[0]

    def load_rasters(self, raster_paths, invert=False):
        rasters = []
        for path in raster_paths:
            with rasterio.open(path) as src:
                raster_data = src.read(1).astype(np.float32)
                if self.original_shape is None:
                    self.original_shape = raster_data.shape
                else:
                    assert self.original_shape == raster_data.shape
                if invert:
                    raster_data = -raster_data
                rasters.append(raster_data)
                if not self.profiles:
                    self.profiles.append(src.profile)
        return rasters

    def compute_percentiles(self, array):
        flat = array[~np.isnan(array)]
        ranks = rankdata(flat, method="average")
        percentiles = (ranks - 1) / (len(flat) - 1)
        result = np.full_like(array, np.nan, dtype=np.float32)
        result[~np.isnan(array)] = percentiles
        return result

    def prepare_objective_array(self):
        objectives = []
        for raster in self.rasters:
            flat = raster.flatten()
            percentiles = self.compute_percentiles(flat)  # Use percentiles instead
            objectives.append(percentiles)
        self.objective_values = np.column_stack(objectives)[self.valid_indices]
        self.original_indices = np.where(self.valid_indices)[0]
    
    def pareto_rank(self):
        """Compute Pareto ranks using FindParetoRanks function"""
        ranks, parentIds = FindParetoRanks(self.objective_values)
        self.parentIds = parentIds
        return ranks
    
    def export_results(self, ranks, actual_bounds, ref_profile, output_dir, 
                      raster_display_names, minimize_flags, use_custom_reference):
        """Export results to GeoDataFrame"""
        # Create full rank raster
        rank_raster = np.full(self.original_shape, np.nan, dtype=np.float32)
        rank_raster.flat[self.original_indices] = ranks
        
        # Get valid points
        valid_mask = ~np.isnan(rank_raster.flatten())
        valid_ranks = rank_raster.flatten()[valid_mask]
        
        # Get spatial coordinates
        rows, cols = np.unravel_index(np.where(valid_mask)[0], self.original_shape)
        transform = ref_profile['transform']
        x_spatial = transform[2] + (cols + 0.5) * transform[0]
        y_spatial = transform[5] + (rows + 0.5) * transform[4]
        
        # Create GeoDataFrame
        gdf = gpd.GeoDataFrame({
            'Rank': valid_ranks,
            'x_spatial': x_spatial,
            'y_spatial': y_spatial,
            'geometry': gpd.points_from_xy(x_spatial, y_spatial)
        }, crs=ref_profile['crs'])
        
        # Add raster values
        for i, (raster, name) in enumerate(zip(self.rasters, raster_display_names)):
            flat_raster = raster.flatten()
            gdf[f'{name} percentile'] = flat_raster[valid_mask]
            # Also add raw values
            for j, raw_raster in enumerate(self.rasters_maximize + self.rasters_minimize):
                if i == j:
                    gdf[f'{name} raw'] = raw_raster.flatten()[valid_mask]
                    break
        
        # Transform to lat/lon
        gdf_latlon = gdf.to_crs('EPSG:4326')
        gdf['lon'] = gdf_latlon.geometry.x
        gdf['lat'] = gdf_latlon.geometry.y
        
        return gdf
    
    def export_geotiff_and_frontier(self, ranks, actual_bounds, ref_profile, output_dir):
        """Export rank raster and Pareto frontier"""
        # Create full rank raster
        rank_raster = np.full(self.original_shape, np.nan, dtype=np.float32)
        rank_raster.flat[self.original_indices] = ranks
        
        # Export rank raster
        rank_path = os.path.join(output_dir, 'pareto_rank.tif')
        with rasterio.open(rank_path, 'w', **ref_profile) as dst:
            dst.write(rank_raster, 1)
        
        # Create percentile rasters
        percentiles_path = os.path.join(output_dir, 'pareto_percentiles.tif')
        percentile_profile = ref_profile.copy()
        percentile_profile['count'] = len(self.rasters)
        with rasterio.open(percentiles_path, 'w', **percentile_profile) as dst:
            for i, raster in enumerate(self.rasters):
                percentile_raster = self.compute_percentiles(raster)
                dst.write(percentile_raster, i + 1)
        
        # Export Pareto frontier
        frontier_mask = (ranks == 0)
        frontier_indices = self.original_indices[frontier_mask]
        rows, cols = np.unravel_index(frontier_indices, self.original_shape)
        transform = ref_profile['transform']
        x_spatial = transform[2] + (cols + 0.5) * transform[0]
        y_spatial = transform[5] + (rows + 0.5) * transform[4]
        
        frontier_gdf = gpd.GeoDataFrame({
            'Rank': ranks[frontier_mask],
            'x_spatial': x_spatial,
            'y_spatial': y_spatial,
            'geometry': gpd.points_from_xy(x_spatial, y_spatial)
        }, crs=ref_profile['crs'])
        
        # Transform to lat/lon
        frontier_latlon = frontier_gdf.to_crs('EPSG:4326')
        frontier_gdf['lon'] = frontier_latlon.geometry.x
        frontier_gdf['lat'] = frontier_latlon.geometry.y
        
        # Save frontier
        frontier_path = os.path.join(output_dir, 'pareto_frontier.geojson')
        frontier_gdf.to_file(frontier_path, driver='GeoJSON')
        
        return frontier_gdf, rank_path, percentiles_path


def FindParetoRanks(pts, epsilon=1e-4):
    """
    Fast and accurate Pareto ranking algorithm with pre-filtering
    to improve performance for large datasets

    Args:
        pts: numpy array of shape (n_points, n_dimensions)
        epsilon: small value for numerical stability
    """
    import time

    start_time = time.time()
    last_report = start_time

    n_points, n_dims = pts.shape
    print(f"Starting fast Pareto ranking on {n_points} points with {n_dims} dimensions")

    # Sort by sum of objectives (descending for higher is better)
    sum_vals = np.sum(pts, axis=1)
    sorted_indices = np.argsort(-sum_vals)  # Note: DESCENDING order

    # Initialize arrays
    paretoRank = np.full(n_points, n_points, dtype=int)
    parentIds = np.arange(n_points, dtype=int)
    current_rank = 0
    remaining = np.ones(n_points, dtype=bool)

    # Calculate standard deviation as a measure for threshold adjustment
    # std_dev = np.std(sum_vals)
    # dynamic_threshold = 0.1 * std_dev
    last_front_maxs = np.full(n_dims, float('inf'))

    # Process points in descending order of sum (optimization)
    while np.any(remaining):
        # Progress reporting
        current_time = time.time()
        if current_time - last_report > 60:  # Report every 60 seconds
            print(
                f"Rank {current_rank}: {np.sum(~remaining)}/{n_points} points processed ({100*np.sum(~remaining)/n_points:.1f}%)"
            )
            last_report = current_time


        current_front = []

        # Fast pre-filtering: Identify candidate points for this front
        candidates = sorted_indices[remaining[sorted_indices]]

        if len(candidates) > 5000 and current_rank > 0:
            # print(f"Rank {current_rank}: Fast-filtering {len(candidates)} candidates...")
            
            # 🚀 ADAPTIVE MULTI-STRATEGY FILTERING:
            # 1. Top 1000 by sum (always include best overall)
            # 2. Top N per dimension (adaptive based on dimensionality)
            # 3. Combine with deduplication
            
            candidate_set = set()
            
            # Strategy 1: Top by sum (best overall performers) - FIXED
            top_n_sum = 1000
            sum_values = sum_vals[candidates]
            top_sum_indices = np.argsort(-sum_values)[:top_n_sum]
            candidate_set.update(candidates[top_sum_indices])
            
            # Strategy 2: Top per dimension (dimension specialists)
            target_total_per_dim = 1000
            top_n_dim = max(100, target_total_per_dim // n_dims) 

            for dim in range(n_dims):
                dim_values = pts[candidates, dim]
                top_indices = np.argsort(-dim_values)[:top_n_dim]
                candidate_set.update(candidates[top_indices])
            
            filtered_candidates = np.array(list(candidate_set))
            # print(f"  ✅ filtered {len(filtered_candidates)} unique candidates from both strategies")
            
            # Safety cap
            if len(filtered_candidates) > 10000:
                print(f"  ⚠️ Capping at 10000 (sorted by sum)")
                candidate_sums = sum_vals[filtered_candidates]
                top_indices = np.argsort(-candidate_sums)[:10000]
                filtered_candidates = filtered_candidates[top_indices]
        else:
            filtered_candidates = candidates
            if current_rank == 0:
                print(f"Rank 0: Processing all {len(filtered_candidates)} candidates")

        # 🔧 CRITICAL FIX: Sort filtered_candidates by sum (DESCENDING) 
        # This ensures we check high-sum points FIRST for dominance
        candidate_sums = sum_vals[filtered_candidates]
        sort_order = np.argsort(-candidate_sums)  # Descending order
        filtered_candidates = filtered_candidates[sort_order]
        
        # if current_rank == 0:
            # print(f"  🔄 Sorted {len(filtered_candidates)} candidates by sum (best first)")

        if len(current_front) == 0:
            # First point in front is never dominated
            current_front.append(filtered_candidates[0])
            remaining[filtered_candidates[0]] = False

        # Vectorized dominance check for remaining candidates
        for idx in filtered_candidates[1:]:
            if not remaining[idx]:
                continue
            
            # Get all current front points at once
            front_pts = pts[current_front]  # Shape: (len(front), n_dims)
            candidate_pt = pts[idx]  # Shape: (n_dims,)
            
            # Vectorized dominance check: front >= candidate in ALL dims AND > in ANY dim
            dominates = np.all(front_pts >= candidate_pt - epsilon, axis=1) & \
                        np.any(front_pts > candidate_pt + epsilon, axis=1)
            
            if not np.any(dominates):
                current_front.append(idx)
                remaining[idx] = False
            else:
                # Find first dominating point
                dominator_idx = np.where(dominates)[0][0]
                parentIds[idx] = current_front[dominator_idx]

        # Assign current rank to front and mark as processed
        if current_front:
            paretoRank[current_front] = current_rank
            remaining[current_front] = False
            # print(
            #     f"Rank {current_rank}: Found {len(current_front)} points in this front"
            # )
        else:
            # If no points in front (shouldn't happen with proper filtering)
            print(f"Warning: Empty front at rank {current_rank}. Stopping.")
            break

        # Move to next rank
        current_rank += 1

    total_time = time.time() - start_time
    print(f"Found {current_rank} Pareto ranks in {total_time:.1f} seconds")
    return paretoRank, parentIds


def compute_percentiles(array):
    """Compute percentiles matching main.py"""
    flat = array[~np.isnan(array)]
    ranks = rankdata(flat, method="average")
    percentiles = (ranks - 1) / (len(flat) - 1)
    result = np.full_like(array, np.nan, dtype=np.float32)
    result[~np.isnan(array)] = percentiles
    return result

print("✅ Optimization library loaded")

In [ ]:
# 🌍 WEB APP INTERFACE - Exact replica of index.html structure

# File mappings (using GitHub raw URLs instead of S3)
GITHUB_DATA_FILES = {
    'minerals': {
        'carbonatite_ree': f'{GITHUB_DATA_URL}carbonatite_ree.tif',
        'cd_zinc': f'{GITHUB_DATA_URL}cd_zinc.tif',
        'graphite': f'{GITHUB_DATA_URL}graphite.tif',
        'magmatic_nickel': f'{GITHUB_DATA_URL}magmatic_nickel.tif',
        'mvt_zinc': f'{GITHUB_DATA_URL}mvt_zinc.tif',
        'pegmatite_lithium': f'{GITHUB_DATA_URL}pegmatite_lithium.tif'
    },
    'priorities': {
        'ecoregions_protected_percentage': f'{GITHUB_DATA_URL}ecoregions_protected_percentage.tif',
        'critical_habitat': f'{GITHUB_DATA_URL}critical_habitat.tif',
        'power_grid_cost': f'{GITHUB_DATA_URL}power_grid_cost.tif',
        'road_cost': f'{GITHUB_DATA_URL}road_cost.tif'
    },
    'reference': {
        'cc.7z': f'{GITHUB_DATA_URL}cc.7z',
        'high_infracost.tif': f'{GITHUB_DATA_URL}power_grid_cost.tif'  # Using power_grid as reference
    }
}

# Priority files mapping (matching main.py PRIORITY_FILES)
PRIORITY_FILES = {
    "protected": ("ecoregions_protected_percentage.tif", True),
    "species": ("critical_habitat.tif", True),
    "infra": (["power_grid_cost.tif", "road_cost.tif"], True),
}

# Create interface widgets matching index.html exactly
mineral_select = widgets.Dropdown(
    options=[
        ('Carbonatite-Hosted REE', 'carbonatite_ree'),
        ('Clastic-Dominated (CD) Zinc', 'cd_zinc'),
        ('Graphite', 'graphite'),
        ('Magmatic Nickel', 'magmatic_nickel'),
        ('Mississippi Valley-Type (MVT) Zinc', 'mvt_zinc'),
        ('Pegmatite-Hosted Lithium', 'pegmatite_lithium'),
        ('N/A', 'na')
    ],
    value='carbonatite_ree',
    description='',
    layout=widgets.Layout(width='100%')
)

infrastructure_check = widgets.Checkbox(value=False, description='Infrastructure Costs')
protected_areas_check = widgets.Checkbox(value=False, description='Protected-Area Gap Analysis')
species_risk_check = widgets.Checkbox(value=False, description='Species at Risk Habitats')

# 🔍 NEW: Local file browser system
selected_files_data = {}


# # Update button text based on environment
# Create file chooser widget
file_chooser = FileChooser(
    path=os.getcwd(),  # Start in current directory
    filename='',
    select_desc="Select",
    change_desc="Select",
    show_hidden=False,
    filter_pattern=['*.tif', '*.tiff', '*.geotiff', '*.TIF', '*.TIFF', '*.GEOTIFF']
)


# Keep the existing clear and process buttons
clear_btn = widgets.Button(
    description='Clear',
    button_style='warning',
    layout=widgets.Layout(width='80px', margin='0 0 0 0px')
)

process_btn = widgets.Button(
    description='Process',
    button_style='success',
    layout=widgets.Layout(width='120px')
)

file_display = widgets.HTML(value='<em>No files selected</em>')

# 🆕 ADD: Create separate file chooser and data storage for MINIMIZE files
selected_files_minimize = {}

file_chooser_minimize = FileChooser(
    path=os.getcwd(),
    filename='',
    select_desc="Select",
    change_desc="Select",
    show_hidden=False,
    filter_pattern=['*.tif', '*.tiff', '*.geotiff', '*.TIF', '*.TIFF', '*.GEOTIFF']
)

file_display_minimize = widgets.HTML(value='<em>No files selected</em>')

clear_btn_minimize = widgets.Button(
    description='Clear',
    button_style='warning',
    layout=widgets.Layout(width='80px', margin='0 0 0 0px')
)


progress_bar = widgets.IntProgress(
    value=0, min=0, max=5,
    description='Progress:',
    layout=widgets.Layout(width='100%', display='none')
)

step_counter = widgets.HTML(value='', layout=widgets.Layout(display='none'))
status_output = widgets.Output()
preview_image = widgets.Output()
result_buttons = widgets.VBox(layout=widgets.Layout(display='none'))


def update_file_display_universal():
    """Universal file display for both Colab and local environments"""
    global selected_files_data

    if selected_files_data:
        files = list(selected_files_data.keys())
        if len(files) <= 3:
            file_list = ', '.join(files)
        else:
            file_list = ', '.join(files[:3]) + f' (+{len(files)-3} more)'

        total_size = sum(file_info['size_mb'] for file_info in selected_files_data.values())

        file_display.value = f'''
        <div style="background: #e8f5e8; padding: 8px; border-radius: 4px; border-left: 3px solid #4CAF50;">
            <strong>✅ {len(files)} file(s) to maximize</strong><br>
            {file_list}<br>
            <small>Total size: {total_size:.2f} MB</small>
        </div>
        '''
    else:
        file_display.value = '<em>No files selected</em>'

def clear_selected_files():
    """Clear selected files - universal"""
    global selected_files_data
    selected_files_data = {}
    update_file_display_universal()
    file_chooser.reset()


In [ ]:
# 🔄 CORE PROCESSING FUNCTIONS - Using GitHub instead of S3

def generate_request_id(mineral, priorities, files, aoi_bounds=None):
    """Generate deterministic request ID like app.js"""
    file_hashes = []
    for filename, file_info in files.items():
        content = file_info['content']
        hash_obj = hashlib.sha256(content)
        file_hashes.append(hash_obj.hexdigest())

    file_hashes.sort()  # deterministic order

    signature_payload = {
        'mineral': mineral,
        'priorities': sorted(priorities),
        'fileHashes': file_hashes
    }
    
    # Add AOI to signature if provided
    if aoi_bounds:
        signature_payload['aoi'] = {
            'nw_lat': aoi_bounds['nw_lat'],
            'nw_lon': aoi_bounds['nw_lon'],
            'se_lat': aoi_bounds['se_lat'],
            'se_lon': aoi_bounds['se_lon']
        }

    sig_str = json.dumps(signature_payload, sort_keys=True)
    print("Signature string for hashing:", sig_str)  # Debug log
    sig_hash = hashlib.sha256(sig_str.encode()).hexdigest()
    return sig_hash

def download_from_github(github_url, local_path):
    """Download file from GitHub and handle special cases like .7z extraction"""
    try:
        # Extract the original filename from URL
        original_filename = os.path.basename(github_url)
        raw_name = os.path.splitext(original_filename)[0]
        
        # Create directory
        local_dir = os.path.dirname(local_path)
        os.makedirs(local_dir, exist_ok=True)
        
        # Download the file without SSL verification
        print(f"📥 Downloading {original_filename}...")
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        response = requests.get(github_url, stream=True, timeout=60, verify=False)
        response.raise_for_status()
        
        # Handle .7z files - extract and return shapefile paths
        if original_filename.endswith('.7z'):
            temp_7z_path = os.path.join(local_dir, original_filename)
            with open(temp_7z_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            print(f"📦 Extracting {original_filename}...")
            with py7zr.SevenZipFile(temp_7z_path, mode='r') as archive:
                archive.extractall(path=local_dir)
            
            # Remove the .7z file after extraction
            os.remove(temp_7z_path)
            
            # Return paths to extracted shapefiles
            cc_shp = os.path.join(local_dir, 'cc.shp')
            cc_shx = os.path.join(local_dir, 'cc.shx')
            if os.path.exists(cc_shp) and os.path.exists(cc_shx):
                print(f"✅ Extracted cc.shp and cc.shx from {original_filename}")
                return {'cc.shp': cc_shp, 'cc.shx': cc_shx}
            else:
                print(f"⚠️ Warning: Expected shapefiles not found in {original_filename}")
                return None
        
        # For regular files (.tif), download and rename
        else:
            # Get the display filename (for renamed downloads)
            display_filename = get_display_filename(raw_name)
            renamed_local_path = os.path.join(local_dir, display_filename)
            
            # Download the file
            with open(renamed_local_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            print(f"✅ Downloaded {original_filename} → {display_filename}")
            return renamed_local_path
        
    except Exception as e:
        print(f"❌ Failed to download {github_url}: {e}")
        return None

def get_display_filename(raw_name):
    """Convert raw name to renamed download filename"""
    
    # Map standard files to renamed filenames
    if raw_name == "power_grid_cost":
        return "Powerline_cost.tif"
    elif raw_name == "road_cost":
        return "Road_cost.tif"
    elif raw_name == "ecoregions_protected_percentage":
        return "Percentage_of_ecoregion_protected.tif"
    elif raw_name == "critical_habitat":
        return "Critical_habitat_density.tif"
    elif raw_name == "carbonatite_ree":
        return "Carbonatite_REE_prospectivity.tif"
    elif raw_name == "cd_zinc":
        return "Clastic-dominated_(CD)_zinc_prospectivity.tif"
    elif raw_name == "graphite":
        return "Graphite_prospectivity.tif"
    elif raw_name == "magmatic_nickel":
        return "Magmatic_nickel_prospectivity.tif"
    elif raw_name == "mvt_zinc":
        return "Mississippi_valley-type_(MVT)_zinc_prospectivity.tif"
    elif raw_name == "pegmatite_lithium":
        return "Pegmatite-hosted_lithium_prospectivity.tif"
    else:
        # For any other files, keep original name
        return raw_name

def fetch_rasters(rasters_maximize, rasters_minimize, request_id, 
                  uploaded_files_maximize, uploaded_files_minimize):
    """Fetch and prepare rasters with separate minimize files"""
    
    local_max, local_min = [], []
    temp_dir = os.path.join(WORK_DIR, 'temp', request_id)
    os.makedirs(temp_dir, exist_ok=True)

    # NEW: Use first uploaded file as reference (if available)
    ref_path = None
    ref_profile = None
    ref_bounds = None
    
    # Check if user uploaded files and if no Canada priority files are selected
    has_user_files = len(uploaded_files_maximize) > 0 or len(uploaded_files_minimize) > 0
    use_custom_reference = (
        has_user_files and 
        len(rasters_maximize) == len([k for k in uploaded_files_maximize.keys() if k.lower().endswith(('.tif', '.tiff', '.geotiff'))]) and
        len(rasters_minimize) == len([k for k in uploaded_files_minimize.keys() if k.lower().endswith(('.tif', '.tiff', '.geotiff'))])
    )

    if use_custom_reference:
        print("🔍 Using uploaded file as reference for projection...")
        # Find first user file to use as reference
        uploaded_files = {**uploaded_files_maximize, **uploaded_files_minimize}
        for key in uploaded_files.keys():
            if key.lower().endswith(('.tif', '.tiff', '.geotiff')):
                file_info = uploaded_files[key]
                ref_path = os.path.join(temp_dir, f"_ref_{key}")
                
                with open(ref_path, 'wb') as f:
                    f.write(file_info['content'])
                    f.flush()
                    os.fsync(f.fileno())
                
                with rasterio.open(ref_path) as ref:
                    ref_profile = ref.profile
                    ref_transform = ref.transform
                    ref_crs = ref.crs
                    ref_shape = (ref.height, ref.width)
                    ref_bounds = ref.bounds
                    
                print(f"✅ Using reference CRS: {ref_crs}")
                print(f"✅ Reference shape: {ref_shape}")
                print(f"✅ Reference bounds: {ref_bounds}")
                break
    
    # Fallback to Canada reference if Canada priority files are selected
    if ref_profile is None:
        print("🔍 Using Canada reference...")
        ref_path = os.path.join(temp_dir, 'reference.tif')
        
        # Download from GitHub
        renamed_ref_path = download_from_github(GITHUB_DATA_FILES['reference']['high_infracost.tif'], ref_path)
        
        if renamed_ref_path:
            with rasterio.open(renamed_ref_path) as ref:
                ref_profile = ref.profile
                ref_transform = ref.transform
                ref_crs = ref.crs
                ref_shape = (ref.height, ref.width)
                ref_bounds = ref.bounds
        else:
            raise FileNotFoundError("❌ Failed to download reference file from GitHub")


    # 🔧 HELPER FUNCTION: Reproject any raster to reference CRS/shape
    def reproject_to_reference(source_path, output_path):
        """Reproject any raster to match reference CRS and shape"""
        try:
            with rasterio.open(source_path) as src:
                src_data = src.read(1).astype(np.float32)
                if src.nodata is not None:
                    src_data = np.where(src_data == src.nodata, np.nan, src_data)
                
                data = np.empty((1, ref_shape[0], ref_shape[1]), dtype=np.float32)
                reproject(
                    source=src_data,
                    destination=data[0],
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear,
                    src_nodata=np.nan,
                    dst_nodata=np.nan
                )
            
            profile = ref_profile.copy()
            profile.update(dtype='float32', count=1, nodata=np.nan)
            
            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data)
            
            return True
        except Exception as e:
            print(f"❌ Reprojection failed: {e}")
            return False

    
    # Process maximize rasters
    for key in rasters_maximize:
        fname = os.path.basename(key)
        local_path = os.path.join(temp_dir, fname)

        if key.startswith('http'):
            # GitHub file - download and rename automatically
            renamed_path = download_from_github(key, local_path)
            if renamed_path:
                tmp_reproj = renamed_path + "_reproj.tif"
                if reproject_to_reference(renamed_path, tmp_reproj):
                    os.replace(tmp_reproj, renamed_path)
                    local_max.append(renamed_path)
                    print(f"✅ Reprojected GitHub file: {os.path.basename(renamed_path)}")

        else:
            # User uploaded file
            if key in uploaded_files_maximize:
                file_info = uploaded_files_maximize[key]
                if 'content' in file_info:
                    print(f"🔄 Processing file: {key}")
                    content = file_info['content']
                    tmp_path = os.path.join(temp_dir, f"_temp_{fname}")
                    
                    try:
                        with open(tmp_path, 'wb') as f:
                            f.write(content)
                            f.flush()
                            os.fsync(f.fileno())
                        
                        source_path = tmp_path
                    except Exception as write_error:
                        print(f"❌ Failed to write temporary file: {write_error}")
                        continue

                    try:
                        with rasterio.open(source_path) as src:
                            # Only reproject if CRS differs
                            if src.crs != ref_crs or src.shape != ref_shape:
                                print(f"🔄 Reprojecting {key} from {src.crs} to {ref_crs}")
                                tmp_reproj = local_path + "_reproj.tif"
                                if reproject_to_reference(source_path, tmp_reproj):
                                    os.replace(tmp_reproj, local_path)
                            else:
                                # Same CRS - copy directly
                                import shutil
                                shutil.copy(source_path, local_path)

                        local_max.append(local_path)
                        print(f"✅ Processed user raster: {key}")

                    except Exception as e:
                        print(f"❌ Failed to process {key}: {e}")
                        continue
                    finally:
                        if os.path.exists(tmp_path):
                            try:
                                os.remove(tmp_path)
                            except:
                                pass
    

    # Process minimize rasters
    for key in rasters_minimize:
        fname = os.path.basename(key)
        local_path = os.path.join(temp_dir, fname)

        if key.startswith('http'):
            # GitHub file - download and rename automatically
            renamed_path = download_from_github(key, local_path)
            if renamed_path:
                tmp_reproj = renamed_path + "_reproj.tif"
                if reproject_to_reference(renamed_path, tmp_reproj):
                    os.replace(tmp_reproj, renamed_path)
                    local_min.append(renamed_path)
                    print(f"✅ Reprojected GitHub file: {os.path.basename(renamed_path)}")
        else:
            # 🔧 NEW: Check in uploaded_files_minimize
            if key in uploaded_files_minimize:
                file_info = uploaded_files_minimize[key]
                if 'content' in file_info:
                    print(f"🔄 Processing MINIMIZE file: {key}")
                    content = file_info['content']
                    tmp_path = os.path.join(temp_dir, f"_temp_{fname}")
                    
                    try:
                        with open(tmp_path, 'wb') as f:
                            f.write(content)
                            f.flush()
                            os.fsync(f.fileno())
                        
                        with rasterio.open(tmp_path) as src:
                            if src.crs != ref_crs or src.shape != ref_shape:
                                print(f"🔄 Reprojecting {key} from {src.crs} to {ref_crs}")
                                tmp_reproj = local_path + "_reproj.tif"
                                if reproject_to_reference(tmp_path, tmp_reproj):
                                    os.replace(tmp_reproj, local_path)
                            else:
                                import shutil
                                shutil.copy(tmp_path, local_path)
                        
                        local_min.append(local_path)
                        print(f"✅ Processed MINIMIZE raster: {key}")
                    
                    except Exception as e:
                        print(f"❌ Failed to process {key}: {e}")
                    finally:
                        if os.path.exists(tmp_path):
                            try:
                                os.remove(tmp_path)
                            except:
                                pass

    return local_max, local_min, ref_bounds, ref_profile, use_custom_reference

def get_mineral_display_name(mineral):
    """Convert mineral code to display name"""
    if not mineral:
        return "Custom mineral prospectivity"

    mineral_names = {
        "carbonatite_ree": "Carbonatite REE prospectivity",
        "cd_zinc": "Clastic-dominated (CD) zinc prospectivity",
        "graphite": "Graphite prospectivity",
        "magmatic_nickel": "Magmatic nickel prospectivity",
        "mvt_zinc": "Mississippi valley-type (MVT) zinc prospectivity",
        "pegmatite_lithium": "Pegmatite-hosted lithium prospectivity",
    }
    return mineral_names.get(mineral, mineral.replace("_", " ").title() + " prospectivity")


def get_mineral_short_name(mineral, capitalize_first=False):
    """Short mineral name for titles/captions, with fixed acronym casing"""
    short_names = {
        "carbonatite_ree": "carbonatite REE",
        "cd_zinc": "CD zinc",
        "graphite": "graphite",
        "magmatic_nickel": "magmatic nickel",
        "mvt_zinc": "MVT zinc",
        "pegmatite_lithium": "pegmatite-hosted lithium",
    }
    name = short_names.get(mineral, mineral.replace("_", " ").lower())
    if capitalize_first:
        name = name[0].upper() + name[1:]
    return name

In [ ]:
# 🎨 VISUALIZATION FUNCTIONS - Using GitHub data source

def create_main_visualization(gdf, mineral, priorities, output_dir):
    """Create main visualization exactly matching main.py"""

    # Download and extract Canada boundary shapefiles from GitHub .7z
    cc_shp = os.path.join(output_dir, 'cc.shp')
    cc_shx = os.path.join(output_dir, 'cc.shx')
    
    try:
        os.makedirs(output_dir, exist_ok=True)
        
        # Download cc.7z from GitHub and extract
        shapefile_paths = download_from_github(GITHUB_DATA_FILES['reference']['cc.7z'], output_dir)
        
        if shapefile_paths and isinstance(shapefile_paths, dict):
            # Update paths to the extracted files
            cc_shp = shapefile_paths.get('cc.shp', cc_shp)
            cc_shx = shapefile_paths.get('cc.shx', cc_shx)
            print(f"✅ Downloaded and extracted Canada boundary files")
        else:
            raise FileNotFoundError("❌ Failed to extract shapefiles from cc.7z")
    except Exception as e:
        raise FileNotFoundError(f"❌ Failed to download Canada boundary files: {e}")


    # Create plot exactly like main.py
    fig, ax = plt.subplots(figsize=(18, 16))

    gdf.plot(
        column="Rank",
        cmap="viridis_r",
        ax=ax,
        markersize=1,
        legend=True,
        legend_kwds={
            "label": "Pareto rank",
            "shrink": 0.8,
            "orientation": "vertical",
            "aspect": 20,
            "pad": 0.01,
        },
    )

    # Plot rank 0 points
    rank0 = gdf[gdf["Rank"] == 0]
    if len(rank0) > 0:
        rank0.plot(ax=ax, color="red", markersize=60, edgecolor="white",
        linewidth=1, label="Pareto rank 0 (frontier)")

    # Read and plot Canada boundary
    canada = gpd.read_file(cc_shp)
    canada = canada.set_crs("epsg:3978")
    canada.boundary.plot(ax=ax, color="black", linewidth=0.5, alpha=0.7)

    # Create dynamic title exactly like main.py
    base_title = "Critical mineral priority map (2 km resolution)"

    dynamic_parts = []
    if mineral:
        mineral_display = get_mineral_short_name(mineral)
        dynamic_parts.append(mineral_display)
    if "infra" in priorities:
        dynamic_parts.append("infrastructure cost")
    if "protected" in priorities:
        dynamic_parts.append("protected areas")
    if "species" in priorities:
        dynamic_parts.append("critical habitat")

    if selected_files_data:
            dynamic_parts.append("custom data")

    if dynamic_parts:
        second_line = " - ".join(dynamic_parts)
        full_single_line = f"{base_title} - {second_line}"

        if len(full_single_line) > 60:
            full_title = f"{base_title}\n{second_line}"
            plt.title(full_title, fontsize=30, pad=20)
        else:
            plt.title(full_single_line, fontsize=34, pad=15)
    else:
        plt.title(base_title, fontsize=34, pad=15)

    # Set fonts exactly like main.py
    ax.tick_params(labelsize=24)
    ax.xaxis.get_offset_text().set_fontsize(24)
    ax.yaxis.get_offset_text().set_fontsize(24)

    # Legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=24, loc="lower left", frameon=True)

    # Colorbar
    cb = ax.get_figure().axes[-1]
    cb.yaxis.label.set_size(24)
    cb.tick_params(labelsize=24)

    plt.tight_layout()

    # Save
    plot_path = os.path.join(output_dir, 'result_map.png')
    plt.savefig(plot_path, dpi=300)
    # plt.show()
    plt.close(fig)

    return plot_path

In [ ]:
# Cell: Create Comprehensive Dashboard (COMPLETE FIXED VERSION WITH LOADING)

def create_comprehensive_dashboard(gdf, mineral, priorities, output_dir, optimizer, ref_bounds, ref_profile, use_custom_reference):
    """
    Complete dashboard with all raster processing and interactive features
    """
    
    print("🎨 Creating comprehensive interactive dashboard...")
    
    # Transform to lat/lon for web mapping
    gdf_map = gdf.to_crs('EPSG:4326')
    gdf_map["lon"] = gdf_map.geometry.x
    gdf_map["lat"] = gdf_map.geometry.y
    gdf_map["x_spatial"] = gdf["x_spatial"]
    gdf_map["y_spatial"] = gdf["y_spatial"]
    

    # Use the pre-resolved names that were processed during initial setup
    raster_display_names = optimizer.raster_display_names
    minimize_flags = optimizer.minimize_flags
    # all_raster_files = optimizer.all_raster_files 
    # ------------------
    
    print(f"✅ Using {len(raster_display_names)} pre-resolved raster names")
    print(f"📊 Raster order: {[name[:30] + '...' if len(name) > 30 else name for name in raster_display_names]}")

    for i, display_name in enumerate(raster_display_names):
        if i >= len(optimizer.rasters):
            break
        
        try:
            # 🔧 DIRECT SAMPLING: Use optimizer.rasters (already reprojected 2D array)
            raster_2d = optimizer.rasters[i]  # This is the reprojected 2D array
            
            # Invert back if minimized
            is_minimized = minimize_flags[i] if i < len(minimize_flags) else False
            if is_minimized:
                raster_2d = -raster_2d
            
            # Get raster dimensions
            raster_height, raster_width = raster_2d.shape
            
            # Sample at GeoDataFrame point locations using DIRECT 2D indexing
            sampled_values = []

            for gdf_idx in range(len(gdf)):
                # optimizer.original_indices[gdf_idx] gives the flat index into the raster
                flat_idx = optimizer.original_indices[gdf_idx]
                
                # Convert flat index to 2D coordinates
                row_idx = flat_idx // raster_width
                col = flat_idx % raster_width
                
                # Sample directly from reprojected raster
                if 0 <= row_idx < raster_height and 0 <= col < raster_width:
                    value = raster_2d[row_idx, col]
                else:
                    value = np.nan
                
                sampled_values.append(value)

            data = np.array(sampled_values, dtype=np.float32)
            percentiles = compute_percentiles(data)
            
            gdf_map[display_name] = percentiles
            gdf_map[display_name + " raw"] = data
            
            print(f"✅ Added {display_name} data (minimized: {is_minimized})")
            
        except Exception as e:
            print(f"⚠️ Failed to process {display_name}: {e}")
            continue

    # 🔧 Export clean GeoJSON
    geojson_path = os.path.join(output_dir, 'all_points.geojson')
    
    export_columns = (
        ["Rank", "lon", "lat", "x_spatial", "y_spatial"] +
        raster_display_names +
        [name + " raw" for name in raster_display_names if name + " raw" in gdf_map.columns]
    )
    
    available_columns = [col for col in export_columns if col in gdf_map.columns]
    gdf_export = gdf_map[available_columns].copy()
    
    # Clean data

    for col in gdf_export.columns:
        if col == 'geometry':
            continue
        if gdf_export[col].dtype in ['float64', 'float32', 'int64', 'int32']:
            gdf_export[col] = gdf_export[col].replace([np.inf, -np.inf], np.nan)
            
            if col in ['lon', 'lat']:
                gdf_export[col] = np.clip(gdf_export[col], -180, 180)
            elif not col.startswith('x_spatial') and not col.startswith('y_spatial'):
                max_val = np.abs(gdf_export[col]).max()
                if not np.isnan(max_val) and max_val > 1e10:
                    extreme_mask = np.abs(gdf_export[col]) > 1e10
                    if extreme_mask.any():
                        gdf_export.loc[extreme_mask, col] = gdf_export.loc[extreme_mask, col].apply(
                            lambda x: f"{x:.3e}" if not np.isnan(x) else np.nan
                        )
        
        if col in ['lon', 'lat']:
            gdf_export[col] = np.round(gdf_export[col].astype(np.float64), 6)
        elif col in ['x_spatial', 'y_spatial']:
            gdf_export[col] = np.round(gdf_export[col].astype(np.float64), 0)
        elif col.endswith(' raw'):
            if col != 'geometry' and gdf_export[col].dtype not in ['object', 'string']:
                max_val = np.abs(gdf_export[col]).max()
                if not np.isnan(max_val) and max_val <= 1e10:
                    gdf_export[col] = gdf_export[col].astype(np.float64)
        elif col not in ['geometry', 'Rank']:
            gdf_export[col] = np.round(gdf_export[col].astype(np.float64), 6)
    
    valid_coords_mask = (
        (gdf_export['lon'] >= -180) & (gdf_export['lon'] <= 180) &
        (gdf_export['lat'] >= -90) & (gdf_export['lat'] <= 90) &
        ~np.isnan(gdf_export['lon']) & ~np.isnan(gdf_export['lat']) &
        ~np.isinf(gdf_export['lon']) & ~np.isinf(gdf_export['lat'])
    )
    
    gdf_export = gdf_export.loc[valid_coords_mask].copy()
    
    if not isinstance(gdf_export, gpd.GeoDataFrame):
        gdf_export = gpd.GeoDataFrame(gdf_export, geometry=gdf_map.geometry[valid_coords_mask], crs=gdf_map.crs)
    
    gdf_export.to_file(geojson_path, driver='GeoJSON')
    
    priority_options_js = json.dumps(raster_display_names)
    rank_min = int(gdf["Rank"].min())
    rank_max = int(gdf["Rank"].max())
    
    # Create dynamic title
    base_title = "Critical mineral priority map"

    if not use_custom_reference:
        base_title += " (2 km resolution)"

    dynamic_parts = []
    if mineral:
        dynamic_parts.append(get_mineral_short_name(mineral))
    if "infra" in priorities:
        dynamic_parts.append("infrastructure cost")
    if "protected" in priorities:
        dynamic_parts.append("protected areas")
    if "species" in priorities:
        dynamic_parts.append("critical habitat")
    if selected_files_data:
        dynamic_parts.append("custom data")
    
    if dynamic_parts:
        second_line = " - ".join(dynamic_parts)
        full_title = f"{base_title} - {second_line}" if len(f"{base_title} - {second_line}") <= 60 else f"{base_title}<br>{second_line}"
    else:
        full_title = base_title
    
    center_lon = (ref_bounds.left + ref_bounds.right) / 2
    center_lat = (ref_bounds.bottom + ref_bounds.top) / 2

    # Transform to lat/lon for map center
    transformer = Transformer.from_crs(ref_profile['crs'], 'EPSG:4326', always_xy=True)
    center_lon_4326, center_lat_4326 = transformer.transform(center_lon, center_lat)

    # comprehensive_dashboard - FIXED HTML with working selection rectangle, click handlers, AND LOADING OVERLAY
    dashboard_html = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <title>{full_title.replace("<br>", " - ")}</title>
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
    <link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet" />
    
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{ height: 100%; font-family: Arial, sans-serif; }}
        #loading {{ position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: rgba(255,255,255,0.9); 
                    display: flex; align-items: center; justify-content: center; font-size: 20px; z-index: 10000; }}
        /* NEW: Map loading overlay */
        #map-loading {{ position: absolute; top: 0; left: 0; right: 0; bottom: 0; 
                        background: rgba(255,255,255,0.95); display: none; 
                        align-items: center; justify-content: center; z-index: 2000;
                        flex-direction: column; gap: 15px; }}
        #map-loading.active {{ display: flex; }}
        .spinner {{ border: 4px solid #f3f3f3; border-top: 4px solid #3b82f6; 
                    border-radius: 50%; width: 50px; height: 50px; 
                    animation: spin 1s linear infinite; }}
        @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}
        #header {{ background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); color: white; padding: 15px; 
                   text-align: center; font-size: 20px; font-weight: bold; }}
        #main-container {{ display: flex; height: calc(100% - 60px); }}
        #left-panel {{ width: 45%; display: flex; flex-direction: column; border-right: 2px solid #ccc; }}
        #right-panel {{ width: 55%; position: relative; }}
        #controls {{ padding: 15px; background: #f5f5f5; border-bottom: 1px solid #ddd; }}
        .control-group {{ margin-bottom: 12px; }}
        .control-label {{ display: block; margin-bottom: 5px; font-weight: bold; color: #333; }}
        select {{ width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px; }}
        button {{ width: 100%; padding: 10px; margin-top: 8px; border: none; border-radius: 4px; 
                  cursor: pointer; font-weight: bold; font-size: 14px; }}
        #update-btn {{ background: #4CAF50; color: white; }}
        .btn-secondary {{ background: #2196F3; color: white; }}
        .btn-export {{ background: #FF9800; color: white; margin-top: 5px; }}
        #plot-container {{ flex: 1; padding: 10px; overflow: hidden; }}
        #plotly-div {{ width: 100%; height: 100%; }}
        #map {{ width: 100%; height: 100%; }}
        #info-box {{ position: absolute; top: 10px; left: 10px; background: white; padding: 12px; 
                     border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.2); max-width: 350px; z-index: 1000; }}
        #legend {{ position: absolute; top: 60px; right: 10px; background: white; padding: 10px; font-size: 14px; z-index: 1000; border: 1px solid #ccc; border-radius: 4px; max-width: 200px; }}
        #legend div {{ display: flex; align-items: center; margin-top: 4px; }}
        #legend span {{ display: inline-block; width: 16px; height: 16px; margin-right: 6px; border: 1px solid #999; }}
    </style>
</head>

<body>
    <div id="loading">Loading...</div>
    <div id="header">{full_title.replace("<br>", " - ")}</div>
    
    <div id="main-container">
        <div id="left-panel">
            <div id="controls">
                <div class="control-group">
                    <label class="control-label">X-axis priority</label>
                    <select id="priority1-select"></select>
                </div>
                
                <div class="control-group">
                    <label class="control-label">Y-axis priority</label>
                    <select id="priority2-select"></select>
                </div>
                
                <button id="update-btn">Update plot</button>
                <button class="btn-secondary" id="reset-btn">Reset view</button>
                <button class="btn-export" id="export-csv-btn">Export data to CSV</button>
            </div>
            
            <div id="plot-container">
                <div id="plotly-div"></div>
            </div>
        </div>
        
        <div id="right-panel">
            <div id="map"></div>
            
            <div id="map-loading">
                <div class="spinner"></div>
                <div style="font-size: 18px; color: #333; font-weight: bold;">Updating map...</div>
                
            </div>
            
            <div id="info-box">
                <div id="point-details">Click a point for details</div>
            </div>
            
            <div id="legend">
                <b>Pareto rank legend</b><br>
                <div><span style="background:#440154;"></span> High pareto rank</div>
                <div><span style="background:#3b528b;"></span> •</div>
                <div><span style="background:#21918c;"></span> •</div>
                <div><span style="background:#5ec962;"></span> •</div>
                <div><span style="background:#fde725;"></span> Low pareto rank</div>
                <div><span style="background:#FF0000; border: 2px solid #FFFFFF;"></span> Pareto rank 0 (frontier)</div>
            </div>
        </div>
    </div>

    <script>
        let allData = null;
        let filteredData = null;
        let map = null;
        let currentP1 = null;
        let currentP2 = null;
        const priorities = {priority_options_js};
                

        // NEW: Loading overlay helper functions
        function showMapLoading() {{
            document.getElementById('map-loading').classList.add('active');
        }}
        
        function hideMapLoading() {{
            document.getElementById('map-loading').classList.remove('active');
        }}
        
        
        // CSV Export Function
        function exportToCSV() {{
            if (!filteredData || !filteredData.features || filteredData.features.length === 0) {{
                alert('No data to export. Please ensure you have data loaded.');
                return;
            }}
            
            console.log(`Exporting ${{filteredData.features.length}} features to CSV`);
            
            // Get all property keys from the first feature
            const firstFeature = filteredData.features[0];
            const headers = Object.keys(firstFeature.properties);
            
            // Create CSV content
            let csvContent = "data:text/csv;charset=utf-8,";
            
            // Add header row
            csvContent += headers.join(",") + "\\n";
            
            // Add data rows
            filteredData.features.forEach(feature => {{
                const row = headers.map(header => {{
                    let value = feature.properties[header];
                    
                    // Handle different data types
                    if (value === null || value === undefined) {{
                        return '';
                    }}
                    
                    // Convert to string and escape commas/quotes
                    value = String(value);
                    if (value.includes(',') || value.includes('"') || value.includes('\\n')) {{
                        value = '"' + value.replace(/"/g, '""') + '"';
                    }}
                    
                    return value;
                }});
                csvContent += row.join(",") + "\\n";
            }});
            
            // Create download link
            const encodedUri = encodeURI(csvContent);
            const link = document.createElement("a");
            link.setAttribute("href", encodedUri);
            
            // Generate filename with timestamp
            const timestamp = new Date().toISOString().slice(0, 19).replace(/[:.]/g, '-');
            const filename = `mineral_priority_data_${{timestamp}}.csv`;
            link.setAttribute("download", filename);
            
            // Trigger download
            document.body.appendChild(link);
            link.click();
            document.body.removeChild(link);
            
            console.log(`CSV export completed: ${{filename}}`);
            alert(`Successfully exported ${{filteredData.features.length}} points to ${{filename}}`);
        }}
        
        map = new maplibregl.Map({{
            container: 'map',
            style: {{
                version: 8,
                sources: {{ osm: {{ type: 'raster', tiles: ['https://a.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png'], tileSize: 256 }} }},
                layers: [{{ id: 'osm-background', type: 'raster', source: 'osm' }}]
            }},
            center: [{center_lon_4326}, {center_lat_4326}],
            zoom: 3
        }});
        
        map.addControl(new maplibregl.NavigationControl(), 'bottom-right');
        
        priorities.forEach((p, i) => {{
            const opt1 = document.createElement('option');
            opt1.value = p;
            opt1.textContent = p;
            document.getElementById('priority1-select').appendChild(opt1);
            
            const opt2 = document.createElement('option');
            opt2.value = p;
            opt2.textContent = p;
            document.getElementById('priority2-select').appendChild(opt2);
        }});
        
        if (priorities.length >= 2) {{
            document.getElementById('priority1-select').value = priorities[0];
            document.getElementById('priority2-select').value = priorities[1];
        }}
        
        map.on('load', () => {{
            const dataUrl = window.location.hash.slice(1) ? decodeURIComponent(window.location.hash.slice(1)) : './all_points.geojson';
            fetch(dataUrl)
                .then(r => r.json())


                .then(data => {{
                    console.log('GeoJSON loaded:', data.features.length, 'features');
                    
                    allData = data;
                    filteredData = data;
                    
                    map.addSource('points', {{ type: 'geojson', data: data, promoteId: 'Rank' }});
                    map.addSource('highlight', {{ type: 'geojson', data: {{ type: 'FeatureCollection', features: [] }} }});
                    
                    map.addLayer({{
                        id: 'points-layer',
                        type: 'circle',
                        source: 'points',
                        filter: ['>', ['get', 'Rank'], 0],
                        paint: {{
                            'circle-radius': ['interpolate', ['linear'], ['zoom'], 3, 2, 10, 6],
                            'circle-color': [
                                'interpolate', ['linear'], ['get', 'Rank'],
                                {rank_min + 1}, '#fde725',
                                {int(rank_min + (rank_max - rank_min) * 0.25)}, '#5ec962',
                                {int(rank_min + (rank_max - rank_min) * 0.5)}, '#21918c',
                                {int(rank_min + (rank_max - rank_min) * 0.75)}, '#3b528b',
                                {rank_max}, '#440154'
                            ],
                            'circle-opacity': 0.7
                        }}
                    }});
                    
                    map.addLayer({{
                        id: 'frontier-layer',
                        type: 'circle',
                        source: 'points',
                        filter: ['==', ['get', 'Rank'], 0],
                        paint: {{
                            'circle-radius': ['interpolate', ['linear'], ['zoom'], 3, 6, 10, 7],
                            'circle-color': '#FF0000',
                            'circle-opacity': 1,
                            'circle-stroke-width': 2,
                            'circle-stroke-color': '#FFFFFF'
                        }}
                    }});
                    
                    map.addLayer({{
                        id: 'highlight-layer',
                        type: 'circle',
                        source: 'highlight',
                        paint: {{
                            'circle-radius': ['interpolate', ['linear'], ['zoom'], 3, 6, 10, 12],
                            'circle-color': 'rgba(0,0,0,0)',
                            'circle-stroke-width': 3,
                            'circle-stroke-color': '#000000'
                        }}
                    }});
                    
                    map.on('click', (e) => {{
                        const features = map.queryRenderedFeatures(e.point, {{
                            layers: ['points-layer', 'frontier-layer']
                        }});
                        
                        if (features.length > 0) {{
                            const feature = features[0];
                            
                            // Smart number formatter function
                            function formatRawValue(value) {{
                                if (value === undefined || value === null || value === "N/A") {{
                                    return "N/A";
                                }}
                                
                                const num = parseFloat(value);
                                if (isNaN(num)) return "N/A";
                                
                                const absNum = Math.abs(num);
                                
                                // Handle zero
                                if (absNum === 0) return "0";
                                
                                // For very small numbers (< 0.01) or very large numbers (> 1000000)
                                if (absNum < 0.01 || absNum > 1000000) {{
                                    // Convert to scientific notation with 2 significant digits
                                    const exponent = Math.floor(Math.log10(absNum));
                                    const mantissa = num / Math.pow(10, exponent);
                                    return `${{mantissa.toFixed(2)}}×10<sup>${{exponent}}</sup>`;
                                }}
                                
                                // For normal range numbers (0.01 to 1,000,000)
                                // Round to 2 decimal places
                                return num.toFixed(2);
                            }}
                            
                            updatePointInfo(feature.properties);
                            highlightPoint(feature);
                            highlightOnPlot(feature);
                        }}
                    }});
                    
                    map.on('mouseenter', 'points-layer', () => {{ map.getCanvas().style.cursor = 'pointer'; }});
                    map.on('mouseleave', 'points-layer', () => {{ map.getCanvas().style.cursor = ''; }});
                    map.on('mouseenter', 'frontier-layer', () => {{ map.getCanvas().style.cursor = 'pointer'; }});
                    map.on('mouseleave', 'frontier-layer', () => {{ map.getCanvas().style.cursor = ''; }});
                    
                    updatePlot();
                    document.getElementById('loading').style.display = 'none';
                }})
                .catch(error => {{
                    console.error('Error:', error);
                    document.getElementById('loading').innerHTML = 'Error: ' + error.message;
                }});
        }});
        
        function updatePlot() {{
            const p1 = document.getElementById('priority1-select').value;
            const p2 = document.getElementById('priority2-select').value;
            if (p1 === p2) {{ alert('Select different priorities'); return; }}
            
            currentP1 = p1;
            currentP2 = p2;
            
            const frontierX = [], frontierY = [], frontierIdx = [];
            const otherX = [], otherY = [], otherRanks = [], otherIdx = [];
            
            filteredData.features.forEach((f, i) => {{
                const x = f.properties[p1];
                const y = f.properties[p2];
                const rank = f.properties.Rank;
                
                if (x != null && y != null) {{
                    if (rank === 0) {{
                        frontierX.push(x);
                        frontierY.push(y);
                        frontierIdx.push(i);
                    }} else {{
                        otherX.push(x);
                        otherY.push(y);
                        otherRanks.push(rank);
                        otherIdx.push(i);
                    }}
                }}
            }});
            
            const traces = [
                {{ x: otherX, y: otherY, mode: 'markers', type: 'scattergl', name: 'Points',
                   marker: {{ size: 4, color: otherRanks, colorscale: 'Viridis', reversescale: true, 
                   cmin: {rank_min + 1}, cmax: {rank_max}, opacity: 0.6 }},
                   customdata: otherIdx, hoverinfo: 'none' }},
                {{ x: frontierX, y: frontierY, mode: 'markers', type: 'scattergl', name: 'Frontier',
                   marker: {{ size: 12, color: '#FF0000', opacity: 1, line: {{ width: 2, color: '#FFFFFF' }}  }},
                   customdata: frontierIdx, hoverinfo: 'none' }}
            ];

            // 🔥 FIX: Check if there's a highlighted feature from the map
            const highlightSource = map.getSource('highlight');
            if (highlightSource) {{
                const highlightData = highlightSource._data;
                if (highlightData && highlightData.features && highlightData.features.length > 0) {{
                    const highlightedFeature = highlightData.features[0];
                    const props = highlightedFeature.properties;
                    
                    // 🔥 ADD: Re-add the highlight on the plot with new axes
                    const x = props[p1];
                    const y = props[p2];
                    if (x != null && y != null) {{
                        traces.push({{
                            x: [x], y: [y], mode: 'markers', type: 'scattergl', name: 'Selected',
                            marker: {{ size: 12, color: 'rgba(0,0,0,0)', line: {{ color: '#000000', width: 3 }} }},
                            showlegend: false, hoverinfo: 'none'
                        }});
                    }}
                }}
            }}
            
            Plotly.react('plotly-div', traces, {{
                title: {{
                    text: `${{p1}} vs. ${{p2}}<br><sub style="font-size:12px; color:#666; font-style:italic;">Click and drag in scatter plot to select a region</sub>`,
                    font: {{ size: 14 }}
                }},
                xaxis: {{ title: p1 + ' (percentile)' }}, 
                yaxis: {{ title: p2 + ' (percentile)' }},
                hovermode: false, dragmode: 'select',
                margin: {{ l: 50, r: 20, t: 60, b: 40 }}, autosize: true
            }}, {{ displayModeBar: false }});
            
            document.getElementById('plotly-div').on('plotly_selected', (eventData) => {{
                if (!eventData || !eventData.range) return;
                

                if (eventData.points.length === 1) {{
                    // Single point click - just highlight it, don't filter
                    const idx = eventData.points[0].customdata;
                    if (idx !== undefined && filteredData.features[idx]) {{
                        const feature = filteredData.features[idx];
                        updatePointInfo(feature.properties);
                        highlightPoint(feature);
                        map.flyTo({{ center: feature.geometry.coordinates, zoom: 10 }});
                    }}
                    return; // Don't proceed to filtering
                }}

                map.getSource('highlight').setData({{ type: 'FeatureCollection', features: [] }});


                const xRange = eventData.range.x;
                const yRange = eventData.range.y;
                if (!xRange || !yRange) return;
                
                // 🆕 NEW: Show loading overlay before filtering
                showMapLoading();
                
                // Use setTimeout to allow the loading overlay to render
                setTimeout(() => {{
                    // Manually filter ALL points within bounds
                    const selected = [];
                    filteredData.features.forEach(f => {{
                        const x = f.properties[currentP1];
                        const y = f.properties[currentP2];
                        
                        if (x != null && y != null &&
                            x >= xRange[0] && x <= xRange[1] &&
                            y >= yRange[0] && y <= yRange[1]) {{
                            selected.push(f);
                        }}
                    }});
                    
                    console.log(`Selected ${{selected.length}} points within bounds`);
                    
                    if (selected.length > 0) {{
                        filteredData = {{ type: 'FeatureCollection', features: selected }};
                        map.getSource('points').setData(filteredData);
                        
                        const bounds = new maplibregl.LngLatBounds();
                        selected.forEach(f => bounds.extend(f.geometry.coordinates));
                        
                        // NEW: Hide loading after map updates
                        map.once('idle', () => {{
                            hideMapLoading();
                        }});
                        
                        map.fitBounds(bounds, {{ padding: 50 }});
                        updatePlot();
                    }} else {{
                        // NEW: Hide loading if no points selected
                        hideMapLoading();
                    }}
                }}, 100);
            }});
        }}
        
        document.getElementById('update-btn').addEventListener('click', updatePlot);
        document.getElementById('reset-btn').addEventListener('click', () => {{
            // NEW: Show loading during reset
            showMapLoading();
            
            setTimeout(() => {{
                filteredData = allData;
                map.getSource('points').setData(allData);
                map.getSource('highlight').setData({{ type: 'FeatureCollection', features: [] }});
                updatePlot();
                
                // NEW: Hide loading after map resets
                map.once('idle', () => {{
                    hideMapLoading();
                }});
                
                map.flyTo({{ center: [{center_lon_4326}, {center_lat_4326}], zoom: 3 }});
            }}, 100);
        }});
        
        
        document.getElementById('export-csv-btn').addEventListener('click', exportToCSV);

        function updatePointInfo(props) {{
            // Smart number formatter (same as above)
            function formatRawValue(value) {{
                if (value === undefined || value === null || value === "N/A") {{
                    return "N/A";
                }}
                
                const num = parseFloat(value);
                if (isNaN(num)) return "N/A";
                
                const absNum = Math.abs(num);
                
                if (absNum === 0) return "0";
                
                if (absNum < 0.01 || absNum > 1000000) {{
                    const exponent = Math.floor(Math.log10(absNum));
                    const mantissa = num / Math.pow(10, exponent);
                    return `${{mantissa.toFixed(2)}}×10<sup>${{exponent}}</sup>`;
                }}
                
                return num.toFixed(2);
            }}
            
            let html = `<strong>Rank:</strong> ${{props.Rank}}<br>`;
            priorities.forEach(p => {{
                if (props[p] != null) {{
                    const rawKey = p + ' raw';
                    const raw = props[rawKey];
                    html += `<strong>${{p}}:</strong> ${{props[p].toFixed(3)}} (raw: ${{formatRawValue(raw)}})<br>`;
                }}
            }});
            html += `<strong>Latitude:</strong> ${{parseFloat(props.lat).toFixed(6)}}°<br>`;
            html += `<strong>Longitude:</strong> ${{parseFloat(props.lon).toFixed(6)}}°`;
            document.getElementById('point-details').innerHTML = html;
        }}
        
        function highlightPoint(feature) {{
            map.getSource('highlight').setData({{ type: 'FeatureCollection', features: [feature] }});
        }}
        
        function highlightOnPlot(feature) {{
            const props = feature.properties;
            if (!currentP1 || !currentP2) return;
            const x = props[currentP1];
            const y = props[currentP2];
            if (x != null && y != null) {{
                const plotDiv = document.getElementById('plotly-div');
                const newData = plotDiv.data.filter(trace => trace.name !== 'Selected');
                newData.push({{
                    x: [x], y: [y], mode: 'markers', type: 'scattergl', name: 'Selected',
                    marker: {{ size: 12, color: 'rgba(0,0,0,0)', line: {{ color: '#000000', width: 3 }} }},
                    showlegend: false, hoverinfo: 'none'
                }});
                Plotly.react('plotly-div', newData, plotDiv.layout, {{ displayModeBar: false }});
            }}
        }}
    </script>
</body>
</html>'''
    
    dashboard_path = os.path.join(output_dir, 'interactive_dashboard.html')
    with open(dashboard_path, 'w', encoding='utf-8') as f:
        f.write(dashboard_html)
    
    print(f"✅ Dashboard created: {dashboard_path}")
    return dashboard_path

In [ ]:
# Cell: Create Large Data Dashboard (COMPLETE HTML INLINE CODE)

def create_largedata_dashboard(gdf, mineral, priorities, output_dir, optimizer, ref_bounds, ref_profile, use_custom_reference):
    """
    Dashboard optimized for large datasets using chunked GeoJSON and canvas-based plotting
    """
    
    print("🎨 Creating large data dashboard with chunked loading...")
    
    # Transform to lat/lon for web mapping
    gdf_map = gdf.to_crs('EPSG:4326')
    gdf_map["lon"] = gdf_map.geometry.x
    gdf_map["lat"] = gdf_map.geometry.y
    gdf_map["x_spatial"] = gdf["x_spatial"]
    gdf_map["y_spatial"] = gdf["y_spatial"]
    
    
    

    raster_display_names = optimizer.raster_display_names
    minimize_flags = optimizer.minimize_flags   
    #----------------------------
    
    # Process rasters (same as before)
    for i, display_name in enumerate(raster_display_names):
        if i >= len(optimizer.rasters):
            break
        
        try:
            raster_2d = optimizer.rasters[i]
            
            is_minimized = minimize_flags[i] if i < len(minimize_flags) else False
            if is_minimized:
                raster_2d = -raster_2d
            
            raster_height, raster_width = raster_2d.shape
            
            sampled_values = []
            for gdf_idx in range(len(gdf)):
                flat_idx = optimizer.original_indices[gdf_idx]
                row_idx = flat_idx // raster_width
                col = flat_idx % raster_width
                
                if 0 <= row_idx < raster_height and 0 <= col < raster_width:
                    value = raster_2d[row_idx, col]
                else:
                    value = np.nan
                
                sampled_values.append(value)
            
            data = np.array(sampled_values, dtype=np.float32)
            percentiles = compute_percentiles(data)
            
            gdf_map[display_name] = percentiles
            gdf_map[display_name + " raw"] = data
            
            print(f"✅ Added {display_name} data (minimized: {is_minimized})")
            
        except Exception as e:
            print(f"⚠️ Failed to process {display_name}: {e}")
            continue
    
    # 🔥 Split GeoJSON into chunks
    print("🔪 Splitting GeoJSON into chunks for efficient loading...")
    
    export_columns = (
        ["Rank", "lon", "lat", "x_spatial", "y_spatial"] +
        raster_display_names +
        [name + " raw" for name in raster_display_names if name + " raw" in gdf_map.columns]
    )
    
    available_columns = [col for col in export_columns if col in gdf_map.columns]
    gdf_export = gdf_map[available_columns].copy()
    
    # Clean data     
    
    for col in gdf_export.columns:
        if col == 'geometry':
            continue
        if gdf_export[col].dtype in ['float64', 'float32', 'int64', 'int32']:
            gdf_export[col] = gdf_export[col].replace([np.inf, -np.inf], np.nan)
            
            if col in ['lon', 'lat']:
                gdf_export[col] = np.clip(gdf_export[col], -180, 180)
            elif not col.startswith('x_spatial') and not col.startswith('y_spatial'):
                max_val = np.abs(gdf_export[col]).max()
                if not np.isnan(max_val) and max_val > 1e10:
                    extreme_mask = np.abs(gdf_export[col]) > 1e10
                    if extreme_mask.any():
                        gdf_export.loc[extreme_mask, col] = gdf_export.loc[extreme_mask, col].apply(
                            lambda x: f"{x:.3e}" if not np.isnan(x) else np.nan
                        )
        
        if col in ['lon', 'lat']:
            gdf_export[col] = np.round(gdf_export[col].astype(np.float64), 6)
        elif col in ['x_spatial', 'y_spatial']:
            gdf_export[col] = np.round(gdf_export[col].astype(np.float64), 0)
        elif col.endswith(' raw'):
            if col != 'geometry' and gdf_export[col].dtype not in ['object', 'string']:
                max_val = np.abs(gdf_export[col]).max()
                if not np.isnan(max_val) and max_val <= 1e10:
                    gdf_export[col] = gdf_export[col].astype(np.float64)
        elif col not in ['geometry', 'Rank']:
            gdf_export[col] = np.round(gdf_export[col].astype(np.float64), 6)
    
    valid_coords_mask = (
        (gdf_export['lon'] >= -180) & (gdf_export['lon'] <= 180) &
        (gdf_export['lat'] >= -90) & (gdf_export['lat'] <= 90) &
        ~np.isnan(gdf_export['lon']) & ~np.isnan(gdf_export['lat']) &
        ~np.isinf(gdf_export['lon']) & ~np.isinf(gdf_export['lat'])
    )
    
    gdf_export = gdf_export.loc[valid_coords_mask].copy()
    
    if not isinstance(gdf_export, gpd.GeoDataFrame):
        gdf_export = gpd.GeoDataFrame(gdf_export, geometry=gdf_map.geometry[valid_coords_mask], crs=gdf_map.crs)
    
    # 🔥 Split into chunks and save
    chunk_size = 100000  # 100k features per chunk
    features = json.loads(gdf_export.to_json())['features']
    num_chunks = (len(features) + chunk_size - 1) // chunk_size
    
    print(f"📦 Creating {num_chunks} chunks ({chunk_size:,} features each)...")
    
    chunk_files = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, len(features))
        
        chunk_data = {
            'type': 'FeatureCollection',
            'features': features[start:end]
        }
        
        chunk_file = f'chunk_{i:04d}.geojson'
        chunk_path = os.path.join(output_dir, chunk_file)
        
        with open(chunk_path, 'w') as f:
            json.dump(chunk_data, f)
        
        chunk_files.append(chunk_file)
        file_size = os.path.getsize(chunk_path) / (1024**2)
        print(f"  ✅ Chunk {i:04d}: {end - start:,} features → {file_size:.1f} MB")
    
    # Create manifest
    manifest = {
        'total_features': len(features),
        'num_chunks': num_chunks,
        'chunk_size': chunk_size,
        'files': chunk_files
    }
    
    manifest_path = os.path.join(output_dir, 'chunks_manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f"✅ Manifest created: {manifest_path}")
    
    # Calculate parameters for HTML
    priority_options_js = json.dumps(raster_display_names)
    rank_min = int(gdf["Rank"].min())
    rank_max = int(gdf["Rank"].max())
    
    # Create dynamic title
    base_title = "Critical mineral priority map"
    if not use_custom_reference:
        base_title += " (2 km resolution)"

    dynamic_parts = []
    if mineral:
        mineral_display = get_mineral_display_name(mineral).replace(" prospectivity", "")
        mineral_display = mineral_display[0].lower() + mineral_display[1:] if mineral_display else mineral_display
        dynamic_parts.append(mineral_display)
    if "infra" in priorities:
        dynamic_parts.append("infrastructure cost")
    if "protected" in priorities:
        dynamic_parts.append("protected areas")
    if "species" in priorities:
        dynamic_parts.append("critical habitat")
    if selected_files_data:
        dynamic_parts.append("custom data")

    if dynamic_parts:
        second_line = " - ".join(dynamic_parts)
        full_title = f"{base_title} - {second_line}" if len(f"{base_title} - {second_line}") <= 60 else f"{base_title}<br>{second_line}"
    else:
        full_title = base_title
    
    center_lon = (ref_bounds.left + ref_bounds.right) / 2
    center_lat = (ref_bounds.bottom + ref_bounds.top) / 2
    transformer = Transformer.from_crs(ref_profile['crs'], 'EPSG:4326', always_xy=True)
    center_lon_4326, center_lat_4326 = transformer.transform(center_lon, center_lat)
    
    # largedata_dashboard - COMPLETE HTML WITH CHUNKED LOADING - Copy entire interactive_dashboard_working.html structure
    dashboard_html = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <title>{full_title.replace("<br>", " - ")}</title>
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    
    <script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
    <link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet" />
    
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{ height: 100%; font-family: Arial, sans-serif; }}
        #loading {{ position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: rgba(255,255,255,0.9); 
                    display: flex; align-items: center; justify-content: center; font-size: 20px; z-index: 10000; }}
        #map-loading {{ position: absolute; top: 0; left: 0; right: 0; bottom: 0; 
                        background: rgba(255,255,255,0.95); display: none; 
                        align-items: center; justify-content: center; z-index: 2000;
                        flex-direction: column; gap: 15px; }}
        #map-loading.active {{ display: flex; }}
        .spinner {{ border: 4px solid #f3f3f3; border-top: 4px solid #3b82f6; 
                    border-radius: 50%; width: 50px; height: 50px; 
                    animation: spin 1s linear infinite; }}
        @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}
        #header {{ background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); color: white; padding: 15px; 
                   text-align: center; font-size: 20px; font-weight: bold; }}
        #main-container {{ display: flex; height: calc(100% - 60px); }}
        #left-panel {{ width: 45%; display: flex; flex-direction: column; border-right: 2px solid #ccc; }}
        #right-panel {{ width: 55%; position: relative; }}
        #controls {{ padding: 15px; background: #f5f5f5; border-bottom: 1px solid #ddd; }}
        .control-group {{ margin-bottom: 12px; }}
        .control-label {{ display: block; margin-bottom: 5px; font-weight: bold; color: #333; }}
        select {{ width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px; }}
        button {{ width: 100%; padding: 10px; margin-top: 8px; border: none; border-radius: 4px; 
                  cursor: pointer; font-weight: bold; font-size: 14px; }}
        #update-btn {{ background: #4CAF50; color: white; }}
        .btn-secondary {{ background: #2196F3; color: white; }}
        .btn-export {{ background: #FF9800; color: white; margin-top: 5px; }}
        #plot-container {{ flex: 0 0 auto; position: relative; background: white; display: grid;
                          grid-template-columns: 60px 1fr; grid-template-rows: 40px minmax(300px, 1fr) 60px;
                          gap: 5px; padding: 20px; overflow: hidden; height: 65vh; }}
        #plot-title {{ grid-column: 1 / 3; grid-row: 1; text-align: center; font-size: 16px; font-weight: bold;
                      display: flex; flex-direction: column; align-items: center; justify-content: center; gap: 2px; }}
        #y-axis-container {{ grid-column: 1; grid-row: 2; display: flex; flex-direction: column;
                            align-items: flex-end; justify-content: space-between; position: relative;
                            padding: 0px 5px 0px 0; }}
        #y-axis-label {{ writing-mode: vertical-rl; text-orientation: mixed;
                        font-weight: bold; font-size: 13px; color: #000; position: absolute;
                        left: 5px; top: 50%; transform: translateY(-50%) rotate(180deg); }}
        .y-tick {{ font-size: 11px; color: #555; text-align: right; padding-right: 5px; width: 35px; }}
        #plot-wrapper {{ grid-column: 2; grid-row: 2; position: relative; background: #f9f9f9; border: 1px solid #ddd; }}
        #plot-canvas {{ width: 100%; height: 100%; display: block; cursor: crosshair; }}
        #x-axis-container {{ grid-column: 2; grid-row: 3; display: flex; flex-direction: column;
                            justify-content: flex-start; position: relative; padding-top: 5px; }}
        #x-ticks-row {{ display: flex; justify-content: space-between; width: 100%; margin-bottom: 5px; }}
        .x-tick {{ font-size: 11px; color: #555; text-align: center; width: -1px; transform: translateX(-50%); }}
        #x-axis-label {{ text-align: center; font-weight: bold; font-size: 13px; color: #000; margin-top: 5px;}}
        .selection-box {{ position: absolute; border: 2px dashed #2196F3; background: rgba(33, 150, 243, 0.1);
                         pointer-events: none; z-index: 50; }}
        #map {{ width: 100%; height: 100%; }}
        #info-box {{ position: absolute; top: 10px; left: 10px; background: white; padding: 12px;
                    border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.2); max-width: 350px; z-index: 1000; }}
        #legend {{ position: absolute; top: 60px; right: 10px; background: white; padding: 10px;
                  font-size: 14px; z-index: 1000; border: 1px solid #ccc; border-radius: 4px; max-width: 200px; }}
        #legend div {{ display: flex; align-items: center; margin-top: 4px; }}
        #legend span {{ display: inline-block; width: 16px; height: 16px; margin-right: 6px; border: 1px solid #999; }}
    </style>
</head>

<body>
    <div id="loading">Loading...</div>
    <div id="header">{full_title.replace("<br>", " - ")}</div>
    
    <div id="main-container">
        <div id="left-panel">
            <div id="controls">
                <div class="control-group">
                    <label class="control-label">X-axis priority</label>
                    <select id="priority1-select"></select>
                </div>
                
                <div class="control-group">
                    <label class="control-label">Y-axis priority</label>
                    <select id="priority2-select"></select>
                </div>
                
                <button id="update-btn">Update plot</button>
                <button class="btn-secondary" id="reset-btn">Reset view</button>
                <button class="btn-export" id="export-csv-btn">Export data to CSV</button>
            
            </div>
            
            <div id="plot-container">
                <div id="plot-title"></div>
                
                <div id="y-axis-container">
                    <div class="y-tick">1</div>
                    <div class="y-tick">0.8</div>
                    <div class="y-tick">0.6</div>
                    <div class="y-tick">0.4</div>
                    <div class="y-tick">0.2</div>
                    <div class="y-tick">0</div>
                    <div id="y-axis-label"></div>
                </div>
                
                <div id="plot-wrapper">
                    <canvas id="plot-canvas"></canvas>
                </div>
                
                <div id="x-axis-container">
                    <div id="x-ticks-row">
                        <div class="x-tick">0</div>
                        <div class="x-tick">0.2</div>
                        <div class="x-tick">0.4</div>
                        <div class="x-tick">0.6</div>
                        <div class="x-tick">0.8</div>
                        <div class="x-tick">1</div>
                    </div>
                    <div id="x-axis-label"></div>
                </div>
            </div>
        </div>
        
        <div id="right-panel">
            <div id="map"></div>
            
            <div id="map-loading">
                <div class="spinner"></div>
                <div style="font-size: 18px; color: #333; font-weight: bold;">Updating map...</div>
            </div>
            
            <div id="info-box">
                <div id="point-details">Click a point for details</div>
            </div>
            
            <div id="legend">
                <b>Pareto rank legend</b><br>
                <div><span style="background:#440154;"></span> High pareto rank</div>
                <div><span style="background:#3b528b;"></span> •</div>
                <div><span style="background:#21918c;"></span> •</div>
                <div><span style="background:#5ec962;"></span> •</div>
                <div><span style="background:#fde725;"></span> Low pareto rank</div>
                <div><span style="background:#FF0000; border: 2px solid #FFFFFF;"></span> Pareto rank 0 (frontier)</div>
            </div>
        </div>
    </div>

    <script>
        let allData = null;
        let filteredData = null;
        let map = null;
        let currentP1 = null;
        let currentP2 = null;
        const priorities = {priority_options_js};
        
        let featureIndex = new Map();
        let currentVisibleFeatures = [];
        
        let canvas = null;
        let ctx = null;
        let plotData = [];
        let plotBounds = {{}};
        let highlightedPoint = null;
        let offscreenCanvas = null;
        let offscreenCtx = null;
        let baseRendered = false;
        
        let isSelecting = false;
        let selectionStart = null;
        let selectionBox = null;
        let justFinishedSelection = false;

        function showMapLoading() {{
            document.getElementById('map-loading').classList.add('active');
        }}
        
        function hideMapLoading() {{
            document.getElementById('map-loading').classList.remove('active');
        }}

        // CSV Export Function
        function exportToCSV() {{
            if (!filteredData || !filteredData.features || filteredData.features.length === 0) {{
                alert('No data to export. Please ensure you have data loaded.');
                return;
            }}
            
            console.log(`Exporting ${{filteredData.features.length}} features to CSV`);
            
            // Get all property keys from the first feature
            const firstFeature = filteredData.features[0];
            const headers = Object.keys(firstFeature.properties);
            
            // Create CSV content
            let csvContent = "data:text/csv;charset=utf-8,";
            
            // Add header row
            csvContent += headers.join(",") + "\\n";
            
            // Add data rows
            filteredData.features.forEach(feature => {{
                const row = headers.map(header => {{
                    let value = feature.properties[header];
                    
                    // Handle different data types
                    if (value === null || value === undefined) {{
                        return '';
                    }}
                    
                    // Convert to string and escape commas/quotes
                    value = String(value);
                    if (value.includes(',') || value.includes('"') || value.includes('\\n')) {{
                        value = '"' + value.replace(/"/g, '""') + '"';
                    }}
                    
                    return value;
                }});
                csvContent += row.join(",") + "\\n";
            }});
            
            // Create download link
            const encodedUri = encodeURI(csvContent);
            const link = document.createElement("a");
            link.setAttribute("href", encodedUri);
            
            // Generate filename with timestamp
            const timestamp = new Date().toISOString().slice(0, 19).replace(/[:.]/g, '-');
            const filename = `mineral_priority_data_${{timestamp}}.csv`;
            link.setAttribute("download", filename);
            
            // Trigger download
            document.body.appendChild(link);
            link.click();
            document.body.removeChild(link);
            
            console.log(`CSV export completed: ${{filename}}`);
            alert(`Successfully exported ${{filteredData.features.length}} points to ${{filename}}`);
        }}
        

        function buildFeatureIndex(features) {{
            console.time('🔍 Building index');
            featureIndex.clear();
            features.forEach(f => {{
                const key = `${{f.properties.lat}},${{f.properties.lon}}`;
                featureIndex.set(key, f);
            }});
            console.timeEnd('🔍 Building index');
            console.log(`✅ Indexed ${{featureIndex.size.toLocaleString()}} features`);
        }}
        
        function getVisibleFeatures() {{
            const zoom = map.getZoom();
            const bounds = map.getBounds();
            
            const west = bounds.getWest();
            const east = bounds.getEast();
            const south = bounds.getSouth();
            const north = bounds.getNorth();
            
            const visible = filteredData.features.filter(f => {{
                const [lon, lat] = f.geometry.coordinates;
                return lon >= west && lon <= east && lat >= south && lat <= north;
            }});
            
            let maxPoints;
            if (zoom < 9) maxPoints = 50000;
            else if (zoom < 11) maxPoints = 100000;
            else maxPoints = 200000;
            
            if (visible.length <= maxPoints) {{
                return visible;
            }}
            
            const frontier = [];
            const others = [];
            
            for (let i = 0; i < visible.length; i++) {{
                const f = visible[i];
                if (f.properties.Rank === 0) {{
                    frontier.push(f);
                }} else {{
                    others.push(f);
                }}
            }}
            
            const step = Math.ceil(others.length / (maxPoints - frontier.length));
            const decimated = [];
            for (let i = 0; i < others.length; i += step) {{
                decimated.push(others[i]);
            }}
            
            return [...frontier, ...decimated];
        }}
        
        let updateMapTimeout = null;
        function updateMapData() {{
            if (!map.getSource('points') || !filteredData) return;
            
            clearTimeout(updateMapTimeout);
            updateMapTimeout = setTimeout(() => {{
                showMapLoading();
                
                requestAnimationFrame(() => {{
                    console.time('🗺️ Update map');
                    currentVisibleFeatures = getVisibleFeatures();
                    
                    map.getSource('points').setData({{
                        type: 'FeatureCollection',
                        features: currentVisibleFeatures
                    }});
                    
                    console.timeEnd('🗺️ Update map');
                    hideMapLoading();
                }});
            }}, 200);
        }}
        
        map = new maplibregl.Map({{
            container: 'map',
            style: {{
                version: 8,
                sources: {{ osm: {{ type: 'raster', tiles: ['https://a.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png'], tileSize: 256 }} }},
                layers: [{{ id: 'osm-background', type: 'raster', source: 'osm' }}]
            }},
            center: [{center_lon_4326}, {center_lat_4326}],
            zoom: 3
        }});
        
        map.addControl(new maplibregl.NavigationControl(), 'bottom-right');
        
        priorities.forEach((p, i) => {{
            const opt1 = document.createElement('option');
            opt1.value = p;
            opt1.textContent = p;
            document.getElementById('priority1-select').appendChild(opt1);
            
            const opt2 = document.createElement('option');
            opt2.value = p;
            opt2.textContent = p;
            document.getElementById('priority2-select').appendChild(opt2);
        }});
        
        if (priorities.length >= 2) {{
            document.getElementById('priority1-select').value = priorities[0];
            document.getElementById('priority2-select').value = priorities[1];
        }}
        
        function rankToColorRGB(rank) {{
            if (rank === 0) return [255, 0, 0];
            
            const t = 1 - (rank / {rank_max});
            let r, g, b;
            
            if (t < 0.25) {{
                const s = t / 0.25;
                r = 68 + (59 - 68) * s;
                g = 1 + (82 - 1) * s;
                b = 84 + (139 - 84) * s;
            }} else if (t < 0.5) {{
                const s = (t - 0.25) / 0.25;
                r = 59 + (33 - 59) * s;
                g = 82 + (145 - 82) * s;
                b = 139 + (140 - 139) * s;
            }} else if (t < 0.75) {{
                const s = (t - 0.5) / 0.25;
                r = 33 + (94 - 33) * s;
                g = 145 + (201 - 145) * s;
                b = 140 + (98 - 140) * s;
            }} else {{
                const s = (t - 0.75) / 0.25;
                r = 94 + (253 - 94) * s;
                g = 201 + (231 - 201) * s;
                b = 98 + (37 - 98) * s;
            }}
            
            return [Math.round(r), Math.round(g), Math.round(b)];
        }}
        
        function renderBasePlot() {{
            if (!offscreenCanvas || !offscreenCtx || plotData.length === 0) return;
            
            console.time('🎨 Render base plot');
            
            const width = offscreenCanvas.width;
            const height = offscreenCanvas.height;
            
            offscreenCtx.clearRect(0, 0, width, height);
            
            const {{ xMin, xMax, yMin, yMax }} = plotBounds;
            const xRange = xMax - xMin;
            const yRange = yMax - yMin;
            
            const imageData = offscreenCtx.createImageData(width, height);
            const data = imageData.data;
            
            const regularPoints = plotData.filter(d => d.rank > 0);
            regularPoints.forEach(d => {{
                const x = Math.round(((d.x - xMin) / xRange) * width);
                const y = Math.round(height - ((d.y - yMin) / yRange) * height);
                
                if (x >= 0 && x < width && y >= 0 && y < height) {{
                    const color = rankToColorRGB(d.rank);
                    const idx = (y * width + x) * 4;
                    
                    for (let dy = -1; dy <= 1; dy++) {{
                        for (let dx = -1; dx <= 1; dx++) {{
                            const px = x + dx;
                            const py = y + dy;
                            if (px >= 0 && px < width && py >= 0 && py < height) {{
                                const i = (py * width + px) * 4;
                                data[i] = color[0];
                                data[i + 1] = color[1];
                                data[i + 2] = color[2];
                                data[i + 3] = 255;
                            }}
                        }}
                    }}
                }}
            }});
            
            offscreenCtx.putImageData(imageData, 0, 0);
            
            const frontierPoints = plotData.filter(d => d.rank === 0);
            frontierPoints.forEach(d => {{
                const x = Math.round(((d.x - xMin) / xRange) * width);
                const y = Math.round(height - ((d.y - yMin) / yRange) * height);
                
                offscreenCtx.fillStyle = 'rgb(255, 0, 0)';
                offscreenCtx.beginPath();
                offscreenCtx.arc(x, y, 5, 0, 2 * Math.PI);  // radius 5 = 10px diameter
                offscreenCtx.fill();
                offscreenCtx.strokeStyle = 'rgb(255, 255, 255)';
                offscreenCtx.lineWidth = 2;
                offscreenCtx.stroke();
            }});
            
            baseRendered = true;
            console.timeEnd('🎨 Render base plot');
        }}
        
        function renderPlot() {{
            if (!canvas || !ctx) return;
            
            if (baseRendered) {{
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                ctx.drawImage(offscreenCanvas, 0, 0);
            }}
            
            if (highlightedPoint && currentP1 && currentP2) {{
                const highlightX = highlightedPoint.properties[currentP1];
                const highlightY = highlightedPoint.properties[currentP2];
                
                if (highlightX != null && highlightY != null) {{
                    const {{ xMin, xMax, yMin, yMax }} = plotBounds;
                    const xRange = xMax - xMin;
                    const yRange = yMax - yMin;
                    
                    const x = ((highlightX - xMin) / xRange) * canvas.width;
                    const y = canvas.height - ((highlightY - yMin) / yRange) * canvas.height;
                    
                    ctx.strokeStyle = '#000000';
                    ctx.lineWidth = 3;
                    ctx.beginPath();
                    ctx.arc(Math.round(x), Math.round(y), 10, 0, 2 * Math.PI);
                    ctx.stroke();
                }}
            }}
        }}
        
        function prepareDataForPlot(features, p1, p2) {{
            const data = [];
            let xMin = Infinity;
            let xMax = -Infinity;
            let yMin = Infinity;
            let yMax = -Infinity;
            
            for (let i = 0; i < features.length; i++) {{
                const f = features[i];
                const x = f.properties[p1];
                const y = f.properties[p2];
                
                if (x != null && y != null) {{
                    data.push({{
                        x: x,
                        y: y,
                        rank: f.properties.Rank,
                        feature: f
                    }});
                    
                    if (x < xMin) xMin = x;
                    if (x > xMax) xMax = x;
                    if (y < yMin) yMin = y;
                    if (y > yMax) yMax = y;
                }}
            }}
            
            return {{ data, bounds: {{ xMin, xMax, yMin, yMax }} }};
        }}
        
        function updatePlot() {{
            const p1 = document.getElementById('priority1-select').value;
            const p2 = document.getElementById('priority2-select').value;
            if (p1 === p2) {{
                alert('Select different priorities');
                return;
            }}
            
            currentP1 = p1;
            currentP2 = p2;
            
            const {{ data, bounds }} = prepareDataForPlot(filteredData.features, p1, p2);
            
            plotData = data;
            plotBounds = bounds;
            baseRendered = false;
            
            
            document.getElementById('plot-title').innerHTML = 
                `<div>${{p1}} vs. ${{p2}}</div>
                <div style="font-size:12px; color:#666; font-style:italic;">
                    Click and drag in scatter plot to select a region
                </div>`;

            document.getElementById('x-axis-label').textContent = `${{p1}} (percentile)`;
            document.getElementById('y-axis-label').textContent = `${{p2}} (percentile)`;
            
            if (!canvas) {{
                canvas = document.getElementById('plot-canvas');
                ctx = canvas.getContext('2d');

                // Create resize function
                window.resizePlotCanvas = function () {{
                    const rect = canvas.getBoundingClientRect();
                    // Set backing store to match CSS size exactly
                    canvas.width = rect.width;
                    canvas.height = rect.height;

                    // Recreate offscreen canvas with same dimensions
                    offscreenCanvas = document.createElement('canvas');
                    offscreenCanvas.width = rect.width;
                    offscreenCanvas.height = rect.height;
                    offscreenCtx = offscreenCanvas.getContext('2d');

                    // Force rerender
                    baseRendered = false;
                    renderBasePlot();
                    renderPlot();
                }};

                // Initial size
                resizePlotCanvas();

                // Handle window resize and zoom
                window.addEventListener('resize', resizePlotCanvas);

                setupCanvasInteraction();
            }}
            
            renderBasePlot();
            renderPlot();
            
            
        }}
        
        function setupCanvasInteraction() {{
            const wrapper = document.getElementById('plot-wrapper');

            // 🔥 FIX: Ignore clicks after box selection
            if (isSelecting || justFinishedSelection) {{
                justFinishedSelection = false;
                return;
            }}
            
            canvas.addEventListener('click', (e) => {{
                const rect = canvas.getBoundingClientRect();
                const x = e.clientX - rect.left;
                const y = e.clientY - rect.top;
                
                const {{ xMin, xMax, yMin, yMax }} = plotBounds;
                const xRange = xMax - xMin;
                const yRange = yMax - yMin;
                
                const dataX = (x / rect.width) * xRange + xMin;
                const dataY = ((rect.height - y) / rect.height) * yRange + yMin;
                
                let nearest = null;
                let minDist = Infinity;
                
                plotData.forEach(d => {{
                    const dx = (d.x - dataX) / xRange;
                    const dy = (d.y - dataY) / yRange;
                    const dist = dx * dx + dy * dy;
                    
                    if (dist < minDist && dist < 0.001) {{
                        minDist = dist;
                        nearest = d;
                    }}
                }});
                
                if (nearest) {{
                    highlightedPoint = nearest.feature;
                    updatePointInfo(nearest.feature.properties);
                    highlightPoint(nearest.feature);
                    renderPlot();
                    map.flyTo({{ center: nearest.feature.geometry.coordinates, zoom: 10 }});
                }}
            }});
            
            canvas.addEventListener('mousedown', (e) => {{
                if (e.button !== 0) return;
                
                isSelecting = true;
                const rect = canvas.getBoundingClientRect();
                selectionStart = {{
                    x: e.clientX - rect.left,
                    y: e.clientY - rect.top,
                    canvasRect: rect
                }};
                
                selectionBox = document.createElement('div');
                selectionBox.className = 'selection-box';
                selectionBox.style.left = selectionStart.x + 'px';
                selectionBox.style.top = selectionStart.y + 'px';
                wrapper.appendChild(selectionBox);
                
                e.preventDefault();
            }});
            
            document.addEventListener('mousemove', (e) => {{
                if (!isSelecting || !selectionBox || !selectionStart) return;
                
                const rect = selectionStart.canvasRect;
                
                let currentX = e.clientX - rect.left;
                let currentY = e.clientY - rect.top;
                
                currentX = Math.max(0, Math.min(currentX, rect.width));
                currentY = Math.max(0, Math.min(currentY, rect.height));
                
                const width = Math.abs(currentX - selectionStart.x);
                const height = Math.abs(currentY - selectionStart.y);
                const left = Math.min(currentX, selectionStart.x);
                const top = Math.min(currentY, selectionStart.y);
                
                selectionBox.style.left = left + 'px';
                selectionBox.style.top = top + 'px';
                selectionBox.style.width = width + 'px';
                selectionBox.style.height = height + 'px';
            }});
            
            document.addEventListener('mouseup', (e) => {{
                if (!isSelecting || !selectionBox || !selectionStart) return;
                
                const rect = selectionStart.canvasRect;
                
                let endX = e.clientX - rect.left;
                let endY = e.clientY - rect.top;
                
                endX = Math.max(0, Math.min(endX, rect.width));
                endY = Math.max(0, Math.min(endY, rect.height));
                
                const {{ xMin, xMax, yMin, yMax }} = plotBounds;
                const xRange = xMax - xMin;
                const yRange = yMax - yMin;
                
                const bbox = {{
                    xMin: (Math.min(selectionStart.x, endX) / rect.width) * xRange + xMin,
                    xMax: (Math.max(selectionStart.x, endX) / rect.width) * xRange + xMin,
                    yMin: ((rect.height - Math.max(selectionStart.y, endY)) / rect.height) * yRange + yMin,
                    yMax: ((rect.height - Math.min(selectionStart.y, endY)) / rect.height) * yRange + yMin
                }};
                
                if (Math.abs(endX - selectionStart.x) > 5 && Math.abs(endY - selectionStart.y) > 5) {{
                    handleSelection(bbox);

                    justFinishedSelection = true; // 🔥 SET the flag
                    // 🔥 ADD: Reset flag after click event would have fired
                    setTimeout(() => {{
                        justFinishedSelection = false;
                    }}, 100);
                }}
                
                if (selectionBox && selectionBox.parentNode) {{
                    selectionBox.remove();
                }}
                selectionBox = null;
                isSelecting = false;
                selectionStart = null;
            }});
        }}
        
        function handleSelection(bbox) {{
            if (!currentP1 || !currentP2) return;

            // 🔥 ADD: Clear highlight when making a selection
            highlightedPoint = null;
            map.getSource('highlight').setData({{ type: 'FeatureCollection', features: [] }});

            
            
            setTimeout(() => {{
                console.time('🔍 Filter selection');
                
                const selected = [];
                for (let i = 0; i < allData.features.length; i++) {{
                    const f = allData.features[i];
                    const x = f.properties[currentP1];
                    const y = f.properties[currentP2];
                    
                    if (x != null && y != null &&
                        x >= bbox.xMin && x <= bbox.xMax &&
                        y >= bbox.yMin && y <= bbox.yMax) {{
                        selected.push(f);
                    }}
                }}
                
                console.timeEnd('🔍 Filter selection');
                console.log(`✅ Selected ${{selected.length.toLocaleString()}} points`);
                
                if (selected.length > 0) {{
                    filteredData = {{ type: 'FeatureCollection', features: selected }};
                    // 🔧 FIX: Update plotData to show only selected points
                    const {{ data }} = prepareDataForPlot(selected, currentP1, currentP2);
                    plotData = data;
                    baseRendered = false;
                    renderBasePlot();
                    renderPlot();

                    showMapLoading();
                    const bounds = new maplibregl.LngLatBounds();
                    for (let i = 0; i < selected.length; i++) {{
                        bounds.extend(selected[i].geometry.coordinates);
                    }}
                    
                    map.fitBounds(bounds, {{ padding: 50, maxZoom: 10 }});
                    
                    map.once('idle', () => {{
                        hideMapLoading();
                    }});
                }} else {{
                    hideMapLoading();
                }}
            }}, 50);
        }}
        
        map.on('load', async () => {{
            console.log('🔍 Loading chunked GeoJSON...');

            // Get data URLs from hash fragment
            const hashData = window.location.hash.slice(1) ? JSON.parse(decodeURIComponent(window.location.hash.slice(1))) : null;
            const chunksData = hashData ? hashData.chunks : null;
            const manifestUrl = hashData ? hashData.manifest : null;

            function getDataUrl(type) {{
                if (type === 'chunks_manifest' && manifestUrl) {{
                    console.log('Loading manifest from:', manifestUrl);
                    return manifestUrl;
                }}
                return './chunks_manifest.json';
            }}

            function getChunkUrl(filename, index) {{
                if (chunksData) {{
                    const chunk = chunksData.find(c => c.filename === filename);
                    if (chunk) {{
                        console.log(`Loading chunk ${{index}} (${{filename}}) from:`, chunk.url);
                        return chunk.url;
                    }}
                }}
                return `./${{filename}}`;
            }}

            document.getElementById('loading').innerHTML = `
                <div style="text-align: center;">
                    <h2>Loading large dataset...</h2>
                    <div style="width: 80%; max-width: 500px; margin: 20px auto; background: #f0f0f0; border-radius: 10px; overflow: hidden;">
                        <div id="progress-bar" style="width: 0%; height: 30px; background: linear-gradient(90deg, #4CAF50, #45a049); transition: width 0.3s; line-height: 30px; color: white; font-weight: bold;"></div>
                    </div>
                    <p id="progress-text">Loading manifest...</p>
                </div>
            `;
            try {{
                const manifestResp = await fetch(getDataUrl('chunks_manifest'));
                if (!manifestResp.ok) {{
                    throw new Error('Chunks not found');
                }}
                const manifest = await manifestResp.json();
                
                document.getElementById('progress-text').textContent =
                    `Loading ${{manifest.num_chunks}} chunks (${{manifest.total_features.toLocaleString()}} points)...`;
                
                const allFeatures = [];
                
                for (let i = 0; i < manifest.files.length; i++) {{
                    const file = manifest.files[i];
                    const progress = ((i + 1) / manifest.files.length) * 100;
                    
                    document.getElementById('progress-bar').style.width = progress + '%';
                    document.getElementById('progress-bar').textContent = progress.toFixed(0) + '%';
                    document.getElementById('progress-text').textContent =
                        `Loading chunk ${{i + 1}}/${{manifest.files.length}}... (${{allFeatures.length.toLocaleString()}} points loaded)`;
                    
                    const chunkResp = await fetch(getChunkUrl(file, i));
                    const chunk = await chunkResp.json();
                    
                    allFeatures.push(...chunk.features);
                    
                    if (i < manifest.files.length - 1) {{
                        await new Promise(resolve => setTimeout(resolve, 50));
                    }}
                }}
                
                console.log(`✅ All chunks loaded: ${{allFeatures.length.toLocaleString()}} features`);
                
                allData = {{
                    type: 'FeatureCollection',
                    features: allFeatures
                }};
                filteredData = allData;
                
                buildFeatureIndex(allFeatures);
                
                map.addSource('points', {{
                    type: 'geojson',
                    data: {{ type: 'FeatureCollection', features: [] }},
                    promoteId: 'Rank',
                    tolerance: 0.5
                }});
                
                map.addSource('highlight', {{
                    type: 'geojson',
                    data: {{ type: 'FeatureCollection', features: [] }}
                }});
                
                map.addLayer({{
                    id: 'points-layer',
                    type: 'circle',
                    source: 'points',
                    filter: ['>', ['get', 'Rank'], 0],
                    paint: {{
                        'circle-radius': [
                            'interpolate', ['linear'], ['zoom'],
                            3, 2,
                            8, 4,
                            12, 7
                        ],
                        'circle-color': [
                            'interpolate', ['linear'], ['get', 'Rank'],
                            {rank_min + 1}, '#fde725',
                            {int(rank_min + (rank_max - rank_min) * 0.25)}, '#5ec962',
                            {int(rank_min + (rank_max - rank_min) * 0.5)}, '#21918c',
                            {int(rank_min + (rank_max - rank_min) * 0.75)}, '#3b528b',
                            {rank_max}, '#440154'
                        ],
                        'circle-opacity': 0.7
                    }}
                }});
                
                map.addLayer({{
                    id: 'frontier-layer',
                    type: 'circle',
                    source: 'points',
                    filter: ['==', ['get', 'Rank'], 0],
                    paint: {{
                        'circle-radius': [
                            'interpolate', ['linear'], ['zoom'],
                            3, 6,
                            10, 7
                        ],
                        'circle-color': '#FF0000',
                        'circle-opacity': 1,
                        'circle-stroke-width': 2,
                        'circle-stroke-color': '#FFFFFF'
                    }}
                }});
                
                map.addLayer({{
                    id: 'highlight-layer',
                    type: 'circle',
                    source: 'highlight',
                    paint: {{
                        'circle-radius': ['interpolate', ['linear'], ['zoom'], 3, 8, 10, 14],
                        'circle-color': 'rgba(0,0,0,0)',
                        'circle-stroke-width': 4,
                        'circle-stroke-color': '#000000'
                    }}
                }});
                
                map.on('moveend', updateMapData);
                map.on('zoomend', updateMapData);
                
                map.on('click', (e) => {{
                    const lngLat = e.lngLat;
                    const clickLng = lngLat.lng;
                    const clickLat = lngLat.lat;
                    
                    let nearest = null;
                    let minDist = Infinity;
                    const searchRadius = 0.05;
                    
                    for (let i = 0; i < currentVisibleFeatures.length; i++) {{
                        const f = currentVisibleFeatures[i];
                        const [lng, lat] = f.geometry.coordinates;
                        
                        const dx = lng - clickLng;
                        const dy = lat - clickLat;
                        
                        if (Math.abs(dx) > searchRadius || Math.abs(dy) > searchRadius) continue;
                        
                        const dist = dx * dx + dy * dy;
                        
                        if (dist < minDist) {{
                            minDist = dist;
                            nearest = f;
                        }}
                    }}
                    
                    if (nearest && minDist < (searchRadius * searchRadius)) {{
                        highlightedPoint = nearest;
                        updatePointInfo(nearest.properties);
                        highlightPoint(nearest);
                        renderPlot();
                    }}
                }});
                
                map.on('mouseenter', 'points-layer', () => {{ map.getCanvas().style.cursor = 'pointer'; }});
                map.on('mouseleave', 'points-layer', () => {{ map.getCanvas().style.cursor = ''; }});
                map.on('mouseenter', 'frontier-layer', () => {{ map.getCanvas().style.cursor = 'pointer'; }});
                map.on('mouseleave', 'frontier-layer', () => {{ map.getCanvas().style.cursor = ''; }});
                
                updateMapData();
                updatePlot();
                
                document.getElementById('loading').style.display = 'none';
                
            }} catch (error) {{
                console.error('❌ Error loading chunks:', error);
                document.getElementById('loading').innerHTML = `
                    <div style="color: red; max-width: 600px; margin: 0 auto; text-align: left; background: white; padding: 20px; border-radius: 8px;">
                        <h2>❌ Error Loading Data</h2>
                        <p><strong>${{error.message}}</strong></p>
                    </div>
                `;
            }}
        }});
        
        document.getElementById('update-btn').addEventListener('click', updatePlot);
        
        document.getElementById('reset-btn').addEventListener('click', () => {{
            filteredData = allData;
            highlightedPoint = null;
            map.getSource('highlight').setData({{ type: 'FeatureCollection', features: [] }});
            
            updateMapData();
            updatePlot();
            map.flyTo({{ center: [{center_lon_4326}, {center_lat_4326}], zoom: 3 }});
        }});

        // Connect the CSV export button
        document.getElementById('export-csv-btn').addEventListener('click', exportToCSV);
        
        function formatRawValue(value) {{
            if (value == null || value === "N/A") return "N/A";
            
            const num = parseFloat(value);
            if (isNaN(num)) return "N/A";
            
            const absNum = Math.abs(num);
            if (absNum === 0) return "0";
            
            if (absNum < 0.01 || absNum > 1000000) {{
                const exponent = Math.floor(Math.log10(absNum));
                const mantissa = num / Math.pow(10, exponent);
                return `${{mantissa.toFixed(2)}}×10<sup>${{exponent}}</sup>`;
            }}
            
            return num.toFixed(2);
        }}
        
        function updatePointInfo(props) {{
            const lines = [`<strong>Rank:</strong> ${{props.Rank}}`];
            
            priorities.forEach(p => {{
                if (props[p] != null) {{
                    const rawKey = p + ' raw';
                    const raw = props[rawKey];
                    const formattedRaw = formatRawValue(raw);
                    lines.push(`<strong>${{p}}:</strong> ${{props[p].toFixed(3)}} (raw: ${{formattedRaw}})`);
                }}
            }});
            
            lines.push(`<strong>Latitude:</strong> ${{parseFloat(props.lat).toFixed(6)}}°`);
            lines.push(`<strong>Longitude:</strong> ${{parseFloat(props.lon).toFixed(6)}}°`);
            
            document.getElementById('point-details').innerHTML = lines.join('<br>');
        }}
        
        function highlightPoint(feature) {{
            map.getSource('highlight').setData({{ type: 'FeatureCollection', features: [feature] }});
        }}
    </script>
</body>
</html>'''
    
    dashboard_path = os.path.join(output_dir, 'interactive_dashboard.html')
    with open(dashboard_path, 'w', encoding='utf-8') as f:
        f.write(dashboard_html)
    
    print(f"✅ Large data dashboard created: {dashboard_path}")
    print(f"📦 Chunks: {num_chunks} × ~100k features")
    print(f"💡 Tip: Use 'python serve.py' to test locally")
    
    return dashboard_path

In [ ]:
def run_ranking_notebook(
    mineral=None,
    priorities=None,
    uploaded_files_maximize=None,
    uploaded_files_minimize=None,
    request_id=None,
    aoi_bounds=None
):
    """
    Run mineral ranking analysis with caching support
    Args:
        mineral: Selected mineral commodity
        priorities: List of priority factors to consider
        uploaded_files_maximize: Dict mapping filenames to binary data for maximize files
        uploaded_files_minimize: Dict mapping filenames to binary data for minimize files
        request_id: Cache identifier (hash of inputs)
        aoi_bounds: Optional tuple of (min_lon, max_lon, min_lat, max_lat) for area of interest
    Returns:
        Dict with 'gdf', 'optimizer', 'request_id', 'output_dir', 'files', and 'stats' keys
    """
    start_time = time.time()
    
    # Default values
    priorities = priorities or []
    uploaded_files_maximize = uploaded_files_maximize or {}
    uploaded_files_minimize = uploaded_files_minimize or {}
    
    # Generate cache ID if not provided - must match original logic
    if not request_id:
        # Only hash the UPLOADED custom files (not standard priority files)
        all_uploaded_files = {**uploaded_files_maximize, **uploaded_files_minimize}
        
        # Use the proper generate_request_id function with AOI support
        request_id = generate_request_id(mineral, priorities, all_uploaded_files, aoi_bounds=aoi_bounds)
        print(f"🔑 Generated request_id: {request_id[:16]}...")
        print(f"   Based on: mineral={mineral}, priorities={priorities}")
        if aoi_bounds:
            print(f"   + AOI: ({aoi_bounds['nw_lon']:.4f}, {aoi_bounds['nw_lat']:.4f}) to ({aoi_bounds['se_lon']:.4f}, {aoi_bounds['se_lat']:.4f})")
        if uploaded_files_maximize or uploaded_files_minimize:
            print(f"   + Custom files: {len(uploaded_files_maximize) + len(uploaded_files_minimize)} uploaded")
    
    output_dir = os.path.join(WORK_DIR, request_id)
    os.makedirs(output_dir, exist_ok=True)
    
    # Define file paths for cached results
    frontier_path = os.path.join(output_dir, 'pareto_frontier.geojson')
    rank_path = os.path.join(output_dir, 'pareto_rank.tif')
    all_points_path = os.path.join(output_dir, 'all_points.geojson')
    
    # 🔧 Check for existing cached results
    frontier_exists = os.path.exists(frontier_path)
    rank_exists = os.path.exists(rank_path)
    all_points_exists = os.path.exists(all_points_path)
    
    if frontier_exists and rank_exists:
        print(f"🔍 Cache HIT! Found existing results at: {output_dir}")
    else:
        print(f"🔍 Cache MISS - will run full analysis")
        if not frontier_exists:
            print(f"   Missing: pareto_frontier.geojson")
        if not rank_exists:
            print(f"   Missing: pareto_rank.tif")
        if not all_points_exists:
            print(f"   Missing: all_points.geojson (needed for cache loading)")
    # if os.path.exists(frontier_path) and os.path.exists(rank_path):
    #     print("\n✅ EXISTING RESULTS FOUND - Loading cached results...")
    #     print(f"📂 Loading from: {output_dir}")
        
    #     start_time = time.time()
        
    #     # Load rank raster and frontier
    #     with rasterio.open(rank_path) as src:
    #         rank_raster = src.read(1)
    #         ref_profile = src.profile
    #         bounds = src.bounds
    #         res = src.res[0]
    #         height, width = rank_raster.shape
        
    #     frontier_gdf = gpd.read_file(frontier_path)
        
    #     # Reconstruct GeoDataFrame from raster
    #     valid_mask = ~np.isnan(rank_raster)
    #     ranks = rank_raster[valid_mask]
    #     rows, cols = np.where(valid_mask)
    #     x_spatial = bounds.left + (cols + 0.5) * res
    #     y_spatial = bounds.top - (rows + 0.5) * res
        
    #     gdf = gpd.GeoDataFrame({
    #         'Rank': ranks,
    #         'x_spatial': x_spatial,
    #         'y_spatial': y_spatial,
    #         'geometry': gpd.points_from_xy(x_spatial, y_spatial)
    #     }, crs=ref_profile['crs'])
        
    #     # Try to load from all_points.geojson (has all raster data)
    #     geojson_path = os.path.join(output_dir, 'all_points.geojson')
    #     # Load all_points.geojson to get raster data columns
    #     # if os.path.exists(geojson_path):
    #     #     print("📥 Loading all_points.geojson...")
    #     #     gdf = gpd.read_file(geojson_path)
    #     #     gdf_geojson = gdf
    #     # else:
    #     #     print("⚠️ all_points.geojson not found, using data from raster only")
    #     #     gdf = gpd.GeoDataFrame(gdf, crs=ref_profile['crs'])
    #     #     gdf_geojson = None
        
    #     # Transform to lat/lon for display
    #     gdf_map = gdf.to_crs('EPSG:4326')
    #     gdf_map['lon'] = gdf_map.geometry.x
    #     gdf_map['lat'] = gdf_map.geometry.y
        
    #     # Merge x_spatial and y_spatial back to gdf_map
    #     gdf_map['x_spatial'] = gdf['x_spatial'].values
    #     gdf_map['y_spatial'] = gdf['y_spatial'].values
        
    #     # Create empty optimizer (no raster data needed for display only)
    #     optimizer = MiningSiteOptimization([], [])
    #     optimizer.raster_display_names = []
    #     optimizer.minimize_flags = []
        
    #     # Create minimal optimizer for cached results
    #     # if gdf_geojson is not None:
    #     #     # Extract raster names from columns
    #     #     raster_cols = [col for col in gdf_geojson.columns 
    #     #                   if col not in ['Rank', 'lon', 'lat', 'x_spatial', 'y_spatial', 'geometry']]
    #     #     optimizer.raster_display_names = [col.replace(' raw', '') for col in raster_cols if ' raw' not in col]
    #     #     optimizer.minimize_flags = [False] * len(optimizer.raster_display_names)
        
    #     # Try to generate visualizations if needed
    #     plot_path = os.path.join(output_dir, 'result_map.png')
    #     if not os.path.exists(plot_path):
    #         try:
    #             plot_path = create_main_visualization(gdf_map, mineral, priorities, output_dir)
    #         except Exception as e:
    #             print(f"⚠️ Could not create visualization: {e}")
    #             plot_path = None
        
    #     dashboard_path = os.path.join(output_dir, 'interactive_dashboard.html')
    #     if not os.path.exists(dashboard_path) and gdf_geojson is not None:
    #         try:
    #             actual_bounds = rasterio.coords.BoundingBox(bounds.left, bounds.bottom, bounds.right, bounds.top)
    #             use_custom_reference = False
    #             # Try to create dashboard
    #             if len(gdf_map) > 500000:
    #                 dashboard_path = create_largedata_dashboard(
    #                     gdf_map, mineral, priorities, output_dir, optimizer,
    #                     actual_bounds, ref_profile, use_custom_reference
    #                 )
    #             else:
    #                 dashboard_path = create_comprehensive_dashboard(
    #                     gdf_map, mineral, priorities, output_dir, optimizer,
    #                     actual_bounds, ref_profile, use_custom_reference
    #                 )
    #         except Exception as e:
    #             print(f"⚠️ Could not create dashboard: {e}")
    #             dashboard_path = None if not os.path.exists(dashboard_path) else dashboard_path
        
    #     # Check for existing visualization files
    #     geojson_path = os.path.join(output_dir, 'all_points.geojson')
        
    #     percentiles_path = os.path.join(output_dir, 'pareto_percentiles.tif')
        
    #     print(f"\n✅ Results loaded and dashboards regenerated in {time.time() - start_time:.1f}s")
        
    #     return {
    #         'gdf': gdf,
    #         'optimizer': optimizer,
    #         'request_id': request_id,
    #         'output_dir': output_dir,
    #         'files': {
    #             'plot': plot_path,
    #             'frontier': frontier_path,
    #             'rank': rank_path,
    #             'percentiles': percentiles_path if os.path.exists(percentiles_path) else None,
    #             'dashboard': dashboard_path
    #         },
    #         'stats': {
    #             'total_points': len(gdf),
    #             'frontier_points': len(frontier_gdf),
    #             'max_rank': int(gdf['Rank'].max()) if len(gdf) > 0 else 0,
    #             'processing_time': time.time() - start_time,
    #             'aoi_applied': aoi_bounds is not None,
    #             'loaded_from_cache': True
    #         }
    #     }
    if os.path.exists(frontier_path) and os.path.exists(rank_path):
        print("\n✅ EXISTING RESULTS FOUND - Loading cached results...")
        print(f"📂 Loading from: {output_dir}")
        
        start_time = time.time()
        
        # Load frontier
        print("📥 Loading pareto_frontier.geojson...")
        frontier_gdf = gpd.read_file(frontier_path)
        
        # Remove 'id' column if present (not needed, just line numbers from file generation)
        if 'id' in frontier_gdf.columns:
            frontier_gdf = frontier_gdf.drop(columns=['id'])
        
        # 🎯 SIMPLIFIED: Load all data directly from all_points.geojson (has everything!)
        geojson_path = os.path.join(output_dir, 'all_points.geojson')
        if os.path.exists(geojson_path):
            print("📥 Loading all_points.geojson (contains complete data)...")
            gdf = gpd.read_file(geojson_path)
            
            # Remove 'id' column if present (not needed, just line numbers from file generation)
            if 'id' in gdf.columns:
                gdf = gdf.drop(columns=['id'])
            
            # Get TIF metadata (needed for dashboard bounds and CRS info)
            print("📥 Loading metadata from pareto_rank.tif...")
            with rasterio.open(rank_path) as src:
                ref_profile = src.profile
                bounds = src.bounds
                res = src.res[0]
            
            # Convert back to original CRS if needed (GeoJSON is in EPSG:4326)
            if gdf.crs.to_string() != ref_profile['crs'].to_string():
                print(f"🔄 Converting from {gdf.crs} to {ref_profile['crs']}...")
                gdf = gdf.to_crs(ref_profile['crs'])
                # Update geometry from x_spatial, y_spatial to ensure consistency
                gdf['geometry'] = gpd.points_from_xy(gdf['x_spatial'], gdf['y_spatial'], crs=ref_profile['crs'])
                
                
            # Verify we have raster data columns
            raster_cols = [col for col in gdf.columns 
                          if col not in ['Rank', 'lon', 'lat', 'x_spatial', 'y_spatial', 'geometry']]
            
            if not raster_cols:
                print("⚠️ WARNING: No raster data columns found in all_points.geojson!")
                print("   This file may be from an older version. Consider deleting cache and re-running.")
            else:
                print(f"✅ Loaded {len(gdf):,} points with {len(raster_cols)} raster data columns")
            
            # 🎨 FORCE REGENERATE visualization and dashboard when loading from cache
            print("🎨 Regenerating result map...")
            try:
                plot_path = create_main_visualization(gdf, mineral, priorities, output_dir)
                print(f"✅ Result map regenerated: {plot_path}")
            except Exception as viz_error:
                print(f"⚠️ Failed to regenerate visualization: {viz_error}")
                plot_path = os.path.join(output_dir, 'result_map.png')
                if not os.path.exists(plot_path):
                    plot_path = None
            
            # 🎨 REGENERATE DASHBOARD with updated data
            print("🎨 Regenerating interactive dashboard...")
            try:
                from types import SimpleNamespace
                from rasterio.coords import BoundingBox
                
                # Extract raster names from loaded data (percentile columns only, not raw)
                raster_display_names = [col for col in raster_cols if not col.endswith(' raw')]
                
                # Create minimal optimizer object for dashboard
                optimizer = SimpleNamespace(
                    raster_display_names=raster_display_names,
                    minimize_flags=[False] * len(raster_display_names),
                    rasters=[],
                    original_indices=[]
                )
                
                # Calculate bounds for dashboard
                actual_bounds = BoundingBox(bounds.left, bounds.bottom, bounds.right, bounds.top)
                use_custom_reference = False
                
                # Choose dashboard type based on size
                if len(gdf) > 500000:
                    dashboard_path = create_largedata_dashboard(
                        gdf, mineral, priorities, output_dir, optimizer,
                        actual_bounds, ref_profile, use_custom_reference
                    )
                else:
                    dashboard_path = create_comprehensive_dashboard(
                        gdf, mineral, priorities, output_dir, optimizer,
                        actual_bounds, ref_profile, use_custom_reference
                    )
                print(f"✅ Dashboard regenerated: {dashboard_path}")
            except Exception as dash_error:
                print(f"⚠️ Failed to regenerate dashboard: {dash_error}")
                import traceback
                traceback.print_exc()
                dashboard_path = os.path.join(output_dir, 'interactive_dashboard.html')
                if not os.path.exists(dashboard_path):
                    dashboard_path = None
            
            print(f"✅ Loaded {len(gdf):,} points ({len(frontier_gdf):,} frontier points)")
            
            # Check for existing visualization files
            percentiles_path = os.path.join(output_dir, 'pareto_percentiles.tif')
            
            print(f"✅ Cache loaded in {time.time() - start_time:.1f}s")
            
            # Return results
            return {
                'gdf': gdf,
                'optimizer': optimizer,
                'request_id': request_id,
                'output_dir': output_dir,
                'files': {
                    'plot': plot_path,
                    'frontier': frontier_path,
                    'rank': rank_path,
                    'percentiles': percentiles_path if os.path.exists(percentiles_path) else None,
                    'dashboard': dashboard_path
                },
                'stats': {
                    'total_points': len(gdf),
                    'frontier_points': len(frontier_gdf),
                    'max_rank': int(gdf['Rank'].max()) if len(gdf) > 0 else 0,
                    'processing_time': time.time() - start_time,
                    'aoi_applied': aoi_bounds is not None,
                    'loaded_from_cache': True
                }
            }
        else:
            # all_points.geojson doesn't exist - try loading from chunks (for large datasets)
            print("📦 all_points.geojson not found - checking for chunked files...")
            
            # Look for chunk files
            chunk_files = sorted([f for f in os.listdir(output_dir) if f.startswith('chunk_') and f.endswith('.geojson')])
            
            if chunk_files:
                print(f"📥 Found {len(chunk_files)} chunk files - loading...")
                
                # Load all chunks and combine
                gdfs = []
                for chunk_file in chunk_files:
                    chunk_path = os.path.join(output_dir, chunk_file)
                    chunk_gdf = gpd.read_file(chunk_path)
                    # Remove 'id' column if present (not needed, just line numbers from file generation)
                    if 'id' in chunk_gdf.columns:
                        chunk_gdf = chunk_gdf.drop(columns=['id'])
                    gdfs.append(chunk_gdf)
                
                gdf = pd.concat(gdfs, ignore_index=True)
                gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=gdfs[0].crs)
                
                print(f"✅ Loaded {len(gdf):,} points from {len(chunk_files)} chunks")
                
                # Get TIF metadata
                with rasterio.open(rank_path) as src:
                    ref_profile = src.profile
                    bounds = src.bounds
                
                # Convert back to original CRS if needed (chunks are in EPSG:4326)
                if gdf.crs.to_string() != ref_profile['crs'].to_string():
                    print(f"🔄 Converting from {gdf.crs} to {ref_profile['crs']}...")
                    gdf = gdf.to_crs(ref_profile['crs'])
                    # Update geometry from x_spatial, y_spatial to ensure consistency
                    gdf['geometry'] = gpd.points_from_xy(gdf['x_spatial'], gdf['y_spatial'], crs=ref_profile['crs'])
                    # Ensure it remains a GeoDataFrame with geometry column set
                    gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=ref_profile['crs'])
                
                # Extract raster names
                raster_cols = [col for col in gdf.columns 
                              if col not in ['Rank', 'lon', 'lat', 'x_spatial', 'y_spatial', 'geometry']]
                raster_display_names = [col for col in raster_cols if not col.endswith(' raw')]
                
                # Create minimal optimizer
                from types import SimpleNamespace
                from rasterio.coords import BoundingBox
                
                optimizer = SimpleNamespace(
                    raster_display_names=raster_display_names,
                    minimize_flags=[False] * len(raster_display_names),
                    rasters=[],
                    original_indices=[]
                )
                
                # Force regenerate result map PNG
                print("🎨 Regenerating result map...")
                try:
                    plot_path = create_main_visualization(gdf, mineral, priorities, output_dir)
                    print(f"✅ Result map regenerated: {plot_path}")
                except Exception as viz_error:
                    print(f"⚠️ Failed to regenerate visualization: {viz_error}")
                    plot_path = os.path.join(output_dir, 'result_map.png')
                    if not os.path.exists(plot_path):
                        plot_path = None
                
                # Force regenerate dashboard
                print("📊 Regenerating dashboard...")
                try:
                    # Calculate bounds for dashboard
                    actual_bounds = BoundingBox(bounds.left, bounds.bottom, bounds.right, bounds.top)
                    use_custom_reference = False
                    
                    # Use large data dashboard since chunk loading is for >500k points
                    dashboard_path = create_largedata_dashboard(
                        gdf, mineral, priorities, output_dir, optimizer,
                        actual_bounds, ref_profile, use_custom_reference
                    )
                    print(f"✅ Dashboard regenerated: {dashboard_path}")
                except Exception as dash_error:
                    print(f"⚠️ Failed to regenerate dashboard: {dash_error}")
                    import traceback
                    traceback.print_exc()
                    dashboard_path = os.path.join(output_dir, 'interactive_dashboard.html')
                    if not os.path.exists(dashboard_path):
                        dashboard_path = None
                
                percentiles_path = os.path.join(output_dir, 'pareto_percentiles.tif')
                
                print(f"✅ Cache loaded from chunks in {time.time() - start_time:.1f}s")
                
                return {
                    'gdf': gdf,
                    'optimizer': optimizer,
                    'request_id': request_id,
                    'output_dir': output_dir,
                    'files': {
                        'plot': plot_path,
                        'frontier': frontier_path,
                        'rank': rank_path,
                        'percentiles': percentiles_path if os.path.exists(percentiles_path) else None,
                        'dashboard': dashboard_path
                    },
                    'stats': {
                        'total_points': len(gdf),
                        'frontier_points': len(frontier_gdf),
                        'max_rank': int(gdf['Rank'].max()) if len(gdf) > 0 else 0,
                        'processing_time': time.time() - start_time,
                        'aoi_applied': aoi_bounds is not None,
                        'loaded_from_cache': True
                    }
                }
            else:
                print("⚠️ No chunked files found either - cache incomplete")
                print("   Proceeding with full analysis...")
    
    # If no existing results or incomplete cache, proceed with normal ranking
    print("\n🔄 No existing results found - proceeding with full analysis...")
    #--------------

    # Prepare raster lists using GitHub URLs
    rasters_maximize, rasters_minimize = [], []
    
    if mineral:
        # Get GitHub URL for mineral
        rasters_maximize.append(GITHUB_DATA_FILES['minerals'][mineral])
    
    for p in priorities:
        if p in PRIORITY_FILES:
            files, minimize = PRIORITY_FILES[p]
            if isinstance(files, str):
                files = [files]
            # Convert filenames to GitHub URLs
            github_urls = []
            for f in files:
                filename_without_ext = os.path.splitext(f)[0]
                if filename_without_ext in GITHUB_DATA_FILES['priorities']:
                    github_urls.append(GITHUB_DATA_FILES['priorities'][filename_without_ext])
                elif filename_without_ext in GITHUB_DATA_FILES['reference']:
                    github_urls.append(GITHUB_DATA_FILES['reference'][filename_without_ext])
            (rasters_minimize if minimize else rasters_maximize).extend(github_urls)
            github_urls = []
    # 🔧 CHANGE: Add maximize files
    for filename in uploaded_files_maximize.keys():
        if filename.lower().endswith(('.tif', '.tiff')):
            rasters_maximize.append(filename)
    
    # 🔧 NEW: Add minimize files to minimize list
    for filename in uploaded_files_minimize.keys():
        if filename.lower().endswith(('.tif', '.tiff')):
            rasters_minimize.append(filename)
    
    print(f"📂 Maximize rasters: {len(rasters_maximize)}")
    print(f"📂 Minimize rasters: {len(rasters_minimize)}")
    
    # Fetch rasters (local download or processing)
    local_max, local_min, actual_bounds, ref_profile, use_custom_reference = fetch_rasters(
        rasters_maximize, rasters_minimize, request_id,
        uploaded_files_maximize, uploaded_files_minimize
    )
    
    if not local_max and not local_min:
        print("❌ ERROR: No valid rasters available for analysis")
        return None
    
    print(f"\n🎯 Running multi-objective optimization...")
    
    # Apply AOI filter if specified
    if aoi_bounds:
        # Convert from dict format to min/max coordinates
        min_lon = aoi_bounds['nw_lon']  # West = minimum longitude
        max_lon = aoi_bounds['se_lon']  # East = maximum longitude
        min_lat = aoi_bounds['se_lat']  # South = minimum latitude
        max_lat = aoi_bounds['nw_lat']  # North = maximum latitude
        
        print(f"\n🗺️ Applying Area of Interest filter:")
        print(f"   Longitude: {min_lon:.4f} to {max_lon:.4f}")
        print(f"   Latitude: {min_lat:.4f} to {max_lat:.4f}")
        
        # Apply AOI filtering to the rasters before optimization
        from rasterio.warp import transform_geom
        from shapely.geometry import box
        
        # Create AOI polygon in WGS84
        aoi_geom_4326 = box(min_lon, min_lat, max_lon, max_lat)
        
        # Transform to raster CRS
        aoi_geom_proj = transform_geom(
            'EPSG:4326',
            ref_profile['crs'],
            aoi_geom_4326
        )
        
        # Create mask
        from rasterio.mask import geometry_mask
        from shapely.geometry import shape as shapely_shape
        aoi_shape = shapely_shape(aoi_geom_proj)
        
        # Apply mask to all rasters
        with rasterio.open(local_max[0] if local_max else local_min[0]) as src:
            aoi_mask = geometry_mask(
                [aoi_geom_proj],
                out_shape=(src.height, src.width),
                transform=src.transform,
                invert=True
            )
        
        print(f"   ✅ AOI mask created: {aoi_mask.sum():,} valid pixels")
        
        # Apply mask to each raster
        for path in local_max + local_min:
            with rasterio.open(path, 'r+') as src:
                data = src.read(1)
                data[~aoi_mask] = np.nan
                src.write(data, 1)
    
    # Run optimization
    optimizer = MiningSiteOptimization(local_max, local_min)
    ranks = optimizer.pareto_rank()
    
    print(f"✅ Optimization complete - {optimizer.num_valid_points:,} valid points analyzed")
    print(f"   Pareto frontier: {(ranks == 0).sum():,} points")
    print(f"   Max rank: {int(ranks.max())}")
    
    # Export results
    gdf = optimizer.export_results(
        ranks, actual_bounds, ref_profile, output_dir,
        optimizer.raster_display_names, optimizer.minimize_flags,
        use_custom_reference
    )
    
    # Generate main visualization
    print("\n🎨 Generating result map...")
    plot_path = create_main_visualization(gdf, mineral, priorities, output_dir)
    
    # Export Pareto frontier and raster files
    print("\n📤 Exporting ranking raster and frontier...")
    frontier_gdf, rank_path, percentiles_path = optimizer.export_geotiff_and_frontier(
        ranks, actual_bounds, ref_profile, output_dir
    )
    
    # Generate interactive dashboard
    print("\n🎨 Generating interactive dashboard...")
    if len(gdf) > 500000:
        print(f"📊 Large dataset ({len(gdf):,} points) - using optimized chunked dashboard")
        dashboard_path = create_largedata_dashboard(
            gdf, mineral, priorities, output_dir, optimizer,
            actual_bounds, ref_profile, use_custom_reference
        )
    else:
        dashboard_path = create_comprehensive_dashboard(
            gdf, mineral, priorities, output_dir, optimizer,
            actual_bounds, ref_profile, use_custom_reference
        )
    
    processing_time = time.time() - start_time
    print(f"\n✅ COMPLETE! Processing time: {processing_time:.1f}s")
    print(f"📁 Results saved to: {output_dir}")
    
    return {
        'gdf': gdf,
        'optimizer': optimizer,
        'request_id': request_id,
        'output_dir': output_dir,
        'files': {
            'plot': plot_path,
            'frontier': os.path.join(output_dir, 'pareto_frontier.geojson'),
            'rank': rank_path,
            'percentiles': percentiles_path,
            'dashboard': dashboard_path
        },
        'stats': {
            'total_points': optimizer.num_valid_points,
            'frontier_points': (ranks == 0).sum(),
            'max_rank': int(ranks.max()),
            'processing_time': processing_time,
            'aoi_applied': aoi_bounds is not None,
            'loaded_from_cache': False
        }
    }


In [ ]:
# 🎯 UPLOAD & PROCESS HANDLERS - Updated for local file browsing
def clear_all_results():
    """Clear all previous results and reset the interface"""
    global selected_files_data

    # Clear the results display areas
    with preview_image:
        clear_output()

    # Hide and reset progress indicators
    progress_bar.layout.display = 'none'
    step_counter.layout.display = 'none'
    progress_bar.value = 0
    step_counter.value = ''

    # Hide result buttons
    result_buttons.layout.display = 'none'
    result_buttons.children = []

    # Clear status output
    with status_output:
        clear_output()


def handle_file_selection():
    """Handle file selection with immediate loading - unified for both environments"""
    global selected_files_data, selected_files_minimize
    with status_output:
        if file_chooser.selected is not None:
            file_path = file_chooser.selected
            filename = os.path.basename(file_path)
            
            # Check if already selected
            if filename in selected_files_data:
                print(f"❌ File already selected: {filename}, ")
                print(f"   If you want to add a file with the same name,")
                print(f"   Please rename it first.")
                return
                        
            # 🆕 CHECK 2: Already selected in minimize list
            if filename in selected_files_minimize:
                print(f"❌ ERROR: File already selected in MINIMIZE list: {filename}")
                print(f"   A file cannot be both maximized AND minimized.")
                print(f"   Please remove it from the minimize list first, or choose a different file.")
                return
            
            try:
                # Quick validation using rasterio
                with rasterio.open(file_path) as src:
                    file_size = os.path.getsize(file_path)
                    size_mb = file_size / (1024 * 1024)
                    shape = src.shape
                    crs = src.crs
                    nodata = src.nodata
                
                
                # 🔧 UNIFIED: Load file content immediately for BOTH environments
                with open(file_path, 'rb') as f:
                    content = f.read()
                
                # Store file info with BOTH content and path for compatibility
                selected_files_data[filename] = {
                    'content': content,      # For processing (both environments)
                    'path': file_path,       # For reference
                    'size_mb': size_mb,      # Unified size tracking
                    'shape': shape,
                    'crs': str(crs),
                    'nodata': nodata,
                    'validated': True
                }
                
                # Use unified display update
                update_file_display_universal()
                print(f"✅ Added {filename} ({size_mb:.1f} MB) - ready for processing")
                
            except Exception as e:
                print(f"❌ Invalid raster file: {e}")


# Connect the file chooser to the handler
file_chooser.register_callback(lambda chooser: handle_file_selection())

def handle_clear_files(button):
    """Handle clear files button click"""
    with status_output:
        clear_all_results()
        clear_selected_files()

#---------new section for minimize files

def update_file_display_minimize():
    """Update display for minimize files"""
    global selected_files_minimize
    
    if selected_files_minimize:
        files = list(selected_files_minimize.keys())
        if len(files) <= 3:
            file_list = ', '.join(files)
        else:
            file_list = ', '.join(files[:3]) + f' (+{len(files)-3} more)'
        
        total_size = sum(file_info['size_mb'] for file_info in selected_files_minimize.values())
        
        file_display_minimize.value = f'''
        <div style="background: #ffe8e8; padding: 8px; border-radius: 4px; border-left: 3px solid #f44336;">
            <strong>✅ {len(files)} file(s) to minimize:</strong><br>
            {file_list}<br>
            <small>Total size: {total_size:.2f} MB</small>
        </div>
        '''
    else:
        file_display_minimize.value = '<em>No files selected</em>'

def handle_file_selection_minimize():
    """Handle minimize file selection"""
    global selected_files_data, selected_files_minimize
    
    with status_output:
        if file_chooser_minimize.selected is not None:
            file_path = file_chooser_minimize.selected
            filename = os.path.basename(file_path)
            
            if filename in selected_files_minimize:
                print(f"❌ File already selected: {filename}, ")
                print(f"   If you want to add a file with the same name,")
                print(f"   Please rename it first.")
                return
            
            if filename in selected_files_data:
                print(f"❌ ERROR: File already selected in MAXIMIZE list: {filename}")
                print(f"   A file cannot be both maximized AND minimized.")
                print(f"   Please remove it from the maximize list first, or choose a different file.")
                return
            
            try:
                with rasterio.open(file_path) as src:
                    file_size = os.path.getsize(file_path)
                    size_mb = file_size / (1024 * 1024)
                    shape = src.shape
                    crs = src.crs
                    nodata = src.nodata
                
                with open(file_path, 'rb') as f:
                    content = f.read()
                
                selected_files_minimize[filename] = {
                    'content': content,
                    'path': file_path,
                    'size_mb': size_mb,
                    'shape': shape,
                    'crs': str(crs),
                    'nodata': nodata,
                    'validated': True
                }
                
                update_file_display_minimize()
                print(f"✅ Added {filename} ({size_mb:.1f} MB) - will be MINIMIZED")
                
            except Exception as e:
                print(f"❌ Invalid raster file: {e}")

def clear_minimize_files(button):
    """Clear minimize files"""
    clear_all_results()
    global selected_files_minimize
    selected_files_minimize = {}
    update_file_display_minimize()
    file_chooser_minimize.reset()

# Connect minimize file chooser
file_chooser_minimize.register_callback(lambda chooser: handle_file_selection_minimize())
clear_btn_minimize.on_click(clear_minimize_files)
#------------------------


def handle_process(button):
    """Handle process button click - replicates app.js uploadFiles()"""
    global selected_files_data, selected_files_minimize

    # 🔧 DISABLE BUTTON to prevent double-clicks
    process_btn.disabled = True
    process_btn.description = 'Processing...'

    clear_all_results()

    with status_output:
        clear_output()

        # Update progress
        progress_bar.layout.display = 'block'
        step_counter.layout.display = 'block'
        progress_bar.value = 0
        step_counter.value = '0/5 steps completed'

        try:
            # Get selections
            selected_mineral = mineral_select.value
            if selected_mineral == 'na':
                selected_mineral = None
            selected_priorities = []

            if infrastructure_check.value:
                selected_priorities.append('infra')
            if protected_areas_check.value:
                selected_priorities.append('protected')
            if species_risk_check.value:
                selected_priorities.append('species')

            # Validate files (matching app.js validation)
            uploaded_files_maximize = selected_files_data or {}
            uploaded_files_minimize = selected_files_minimize or {}
            
            if not selected_mineral and not selected_priorities and not uploaded_files_maximize and not uploaded_files_minimize:
                print("❌ Please select at least one option")
                process_btn.disabled = False
                process_btn.description = 'Process'
                return
            
            # Validate both file sets
            total_files = len(uploaded_files_maximize) + len(uploaded_files_minimize)
            if total_files > 20:
                print(f"❌ Too many files ({total_files}). Maximum is 20.")
                process_btn.disabled = False
                process_btn.description = 'Process'
                return


            # This should not happen: Check for files in BOTH maximize and minimize lists
            maximize_filenames = set(uploaded_files_maximize.keys())
            minimize_filenames = set(uploaded_files_minimize.keys())
            duplicate_files = maximize_filenames & minimize_filenames
            
            if duplicate_files:
                print("❌ ERROR: Same file(s) selected for both maximize AND minimize:")
                for filename in sorted(duplicate_files):
                    print(f"   • {filename}")
                print("\n💡 A file cannot be both maximized and minimized.")
                print("   Please remove it from one of the lists.")
                process_btn.disabled = False
                process_btn.description = 'Process'
                return
            
            # Validate file sizes for BOTH sets
            all_uploaded_files = {**uploaded_files_maximize, **uploaded_files_minimize}
            
            for filename, file_info in all_uploaded_files.items():
                if 'content' in file_info:
                    file_size = len(file_info['content'])
                else:
                    print(f"❌ Invalid file info for {filename}")      
                    process_btn.disabled = False
                    process_btn.description = 'Process'
                    return

                if file_size > 2 * 1024 * 1024 * 1024:  # 2GB limit
                    size_gb = file_size / (1024**3)
                    print(f"❌ '{filename}' is too large ({size_gb:.2f} GB). Max 2 GB.")
                    process_btn.disabled = False
                    process_btn.description = 'Process'
                    return
            
            # Get AOI bounds if provided (checking for NaN)
            aoi_bounds = None
            if (not np.isnan(aoi_nw_lat.value) and 
                not np.isnan(aoi_nw_lon.value) and 
                not np.isnan(aoi_se_lat.value) and 
                not np.isnan(aoi_se_lon.value)):
                
                aoi_bounds = {
                    'nw_lat': aoi_nw_lat.value,
                    'nw_lon': aoi_nw_lon.value,
                    'se_lat': aoi_se_lat.value,
                    'se_lon': aoi_se_lon.value
                }
                
                # Validate AOI bounds
                if aoi_bounds['nw_lat'] <= aoi_bounds['se_lat']:
                    print("❌ Invalid AOI: NW latitude must be greater than SE latitude")
                    process_btn.disabled = False
                    process_btn.description = 'Process'
                    return
                
                if aoi_bounds['nw_lon'] >= aoi_bounds['se_lon']:
                    print("❌ Invalid AOI: NW longitude must be less than SE longitude")
                    process_btn.disabled = False
                    process_btn.description = 'Process'
                    return
                print(f"✅ AOI bounds validated: NW({aoi_bounds['nw_lat']:.4f}, {aoi_bounds['nw_lon']:.4f}) - SE({aoi_bounds['se_lat']:.4f}, {aoi_bounds['se_lon']:.4f})")

            # 🆕 OPTIONAL: Add warning if partial values entered
            elif (not np.isnan(aoi_nw_lat.value) or 
                not np.isnan(aoi_nw_lon.value) or 
                not np.isnan(aoi_se_lat.value) or 
                not np.isnan(aoi_se_lon.value)):
                
                print("⚠️ Warning: Partial AOI coordinates detected but not used")


            # NEW: Early duplicate and single-file warnings
            # Build expected display names for all selected data
            all_expected_names = []

            # Add mineral name if selected
            if selected_mineral:
                mineral_display = get_mineral_display_name(selected_mineral)
                all_expected_names.append(mineral_display)

            # Add priority names if selected
            if 'protected' in selected_priorities:
                all_expected_names.append("Percentage of ecoregion protected")
            if 'species' in selected_priorities:
                all_expected_names.append("Critical habitat density")
            if 'infra' in selected_priorities:
                all_expected_names.extend(["Powerline cost", "Road cost"])

            # 🔧 NEW: Add uploaded MAXIMIZE file names
            for filename in uploaded_files_maximize.keys():
                if filename.lower().endswith(('.tif', '.tiff', '.geotiff')):
                    raw_name = os.path.splitext(filename)[0]
                    display_name = raw_name.replace("_", " ")
                    all_expected_names.append(f"{display_name} (maximize)")

            # 🔧 NEW: Add uploaded MINIMIZE file names
            for filename in uploaded_files_minimize.keys():
                if filename.lower().endswith(('.tif', '.tiff', '.geotiff')):
                    raw_name = os.path.splitext(filename)[0]
                    display_name = raw_name.replace("_", " ")
                    all_expected_names.append(f"{display_name} (minimize)")

            print(f"📊 Total expected rasters: {len(all_expected_names)}")

            # Check for duplicates
            name_counts = {}
            duplicates_found = []
            for name in all_expected_names:
                if name in name_counts:
                    name_counts[name] += 1
                    if name not in duplicates_found:
                        duplicates_found.append(name)
                else:
                    name_counts[name] = 1

            # Show duplicate warning if found
            if duplicates_found:
                print("\n❗ DUPLICATE RASTER NAMES DETECTED:")
                for dup_name in duplicates_found:
                    count = name_counts[dup_name]
                    print(f"   • '{dup_name}' appears {count} times")
                print("⚠️   Please make sure these are intentional.")
                print("⚠️  The system will auto-rename duplicates")
                print("🔄 Proceeding with automatic duplicate resolution...\n")

            # Check for single file (cannot do Pareto ranking)
            if len(all_expected_names) <= 1:
                print("\n❌ INSUFFICIENT DATA FOR PARETO RANKING:")
                print(f"   Only {len(all_expected_names)} raster(s) selected")
                print("   Pareto ranking requires at least 2 different priority rasters")
                print("\n💡 Please add more data by:")
                print("   • Selecting additional priority checkboxes")
                print("   • Uploading more custom GeoTIFF files") 
                
                # Re-enable button and stop
                process_btn.disabled = False
                process_btn.description = 'Process'
                return

            print("✅ Pre-analysis verification complete")

            progress_bar.value = 1
            step_counter.value = '1/5 steps completed'
            print("✅ Validation complete")

            progress_bar.value = 1
            step_counter.value = '1/5 steps completed'
            print("✅ Validation complete")

            # Run the analysis
            progress_bar.value = 2
            step_counter.value = '2/5 steps completed'
            print("🚀 Starting analysis...")

            # 🔧 CHANGE: Pass both file sets to run_ranking_notebook
            results = run_ranking_notebook(
                mineral=selected_mineral,
                priorities=selected_priorities,
                uploaded_files_maximize=uploaded_files_maximize,
                uploaded_files_minimize=uploaded_files_minimize,  # NEW parameter
                aoi_bounds=aoi_bounds
            )

            progress_bar.value = 3
            step_counter.value = '3/5 steps completed'
            print("✅ Analysis complete")

            # Display results
            progress_bar.value = 4
            step_counter.value = '4/5 steps completed'
            print("🎨 Generating outputs...")

            with preview_image:
                clear_output()

                # Show plot if exists
                if results['files'].get('plot'):
                    display(Image(filename=results['files']['plot']))
                else:
                    display(HTML('''
                    <div style="background: #fff3cd; padding: 15px; margin: 10px 0; border-radius: 8px; border-left: 4px solid #ffc107;">
                        <p style="margin: 0; color: #856404;">
                            ℹ️ <strong>Custom Reference Used:</strong> Static map visualization skipped.<br>
                            Use the Interactive Dashboard to explore your results.
                        </p>
                    </div>
                    '''))

                # Show summary stats
                stats = results['stats']
                summary_html = f"""
                <div style="background: #f0f8ff; padding: 15px; margin: 10px 0; border-radius: 8px;">
                    <h3>📊 Analysis Results</h3>
                    <p><strong>Total Points:</strong> {stats['total_points']:,}</p>
                    <p><strong>Pareto Frontier Points:</strong> {stats['frontier_points']:,}</p>
                    <p><strong>Maximum Rank:</strong> {stats['max_rank']}</p>
                    <p><strong>Processing Time:</strong> {stats['processing_time']:.1f} seconds</p>
                </div>
                """
                display(HTML(summary_html))

            # Show simple buttons (matching web version)
            download_buttons = show_simple_results_interface(results)
            result_buttons.children = download_buttons
            result_buttons.layout.display = 'flex'

            # Final progress
            progress_bar.value = 5
            step_counter.value = '5/5 steps completed'
            print("✅ All steps complete!")

            # Reset selected files and displays
            selected_files_data = {}
            selected_files_minimize = {}
            update_file_display_universal()
            update_file_display_minimize()
            file_chooser.reset()
            file_chooser_minimize.reset()


            # Show file paths
            print(f"\n📁 Results saved to: {results['output_dir']}")
            for file_type, file_path in results['files'].items():
                if file_path:
                    print(f"  • {file_type}: {os.path.basename(file_path)}")

        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()
        finally:
            # 🔧 ALWAYS RE-ENABLE BUTTON when done (success or error)
            process_btn.disabled = False
            process_btn.description = 'Process'

# Connect buttons to handlers
clear_btn.on_click(handle_clear_files)
process_btn.on_click(handle_process)

In [ ]:
def download_file_simple(file_path):
    """Simple download function for Colab"""
    if IN_COLAB and os.path.exists(file_path):
        try:
            from google.colab import files
            files.download(file_path)
            print(f"✅ Downloaded: {os.path.basename(file_path)}")
        except Exception as e:
            print(f"❌ Download failed: {e}")
    else:
        print(f"📁 File path: {file_path}")

def open_local_folder(file_path):
    """Open folder containing the file in Windows Explorer"""
    import os
    import subprocess
    import platform

    try:
        folder_path = os.path.dirname(file_path)

        if platform.system() == "Windows":
            # Open Windows Explorer and select the file
            subprocess.run(['explorer', '/select,', os.path.abspath(file_path)], check=True)
            print(f"📂 Opened folder: {folder_path}")
            print(f"📄 File highlighted: {os.path.basename(file_path)}")
        elif platform.system() == "Darwin":  # macOS
            subprocess.run(['open', '-R', file_path], check=True)
            print(f"📂 Opened Finder: {folder_path}")
        else:  # Linux
            subprocess.run(['xdg-open', folder_path], check=True)
            print(f"📂 Opened file manager: {folder_path}")

    except Exception as e:
        print(f"❌ Failed to open folder: {e}")
        print(f"📁 Folder path: {folder_path}")
        print(f"📄 File: {os.path.basename(file_path)}")



In [ ]:
# Add this after the visualization functions

# 📖 HOSTING INSTRUCTIONS FOR CLOUD ENVIRONMENTS

def show_hosting_instructions(output_dir):
    """Display instructions for hosting the dashboard locally after download"""
    from IPython.display import display, HTML
    
    # Customize message based on environment
    if CLOUD_ENV_NAME:
        env_message = f"<p style='font-size: 15px; margin: 0 0 15px 0;'>You're using <strong>{CLOUD_ENV_NAME}</strong>, which doesn't support direct web hosting. Follow these steps to view your interactive dashboard:</p>"
    else:
        env_message = "<p style='font-size: 15px; margin: 0 0 15px 0;'>Your environment doesn't support direct web hosting. Follow these steps to view your interactive dashboard:</p>"
    
    instructions_html = f'''
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 25px; margin: 20px 0; border-radius: 12px; box-shadow: 0 8px 20px rgba(102,126,234,0.3);">
        <h2 style="margin-top: 0; text-align: center;">How to View Your Interactive Dashboard</h2>
        
        {env_message}
        
        <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 8px; margin: 15px 0;">
            <h3 style="margin-top: 0; color: #FFD700;">Method 1: Simple Python Server</h3>
            <ol style="line-height: 1.8; font-size: 15px;">
                <li><strong>Download all files</strong> to the same folder on your computer</li>
                <li><strong>Open terminal/command prompt</strong> in that folder:
                    <ul style="margin: 8px 0; font-size: 14px;">
                        <li>Windows: Shift + Right-click → "Open PowerShell window here"</li>
                        <li>Mac: Right-click → Services → "New Terminal at Folder"</li>
                        <li>Linux: Right-click → "Open in Terminal"</li>
                    </ul>
                </li>
                <li><strong>Run this command:</strong>
                    <div style="background: rgba(0,0,0,0.3); padding: 12px; border-radius: 6px; margin: 10px 0; font-family: monospace; font-size: 14px;">
                        <code style="color: #FFD700;">python -m http.server 8000</code>
                    </div>
                </li>
                <li><strong>Open your browser</strong> and go to:
                    <div style="background: rgba(0,0,0,0.3); padding: 12px; border-radius: 6px; margin: 10px 0; font-family: monospace; font-size: 14px;">
                        <code style="color: #FFD700;">http://localhost:8000/interactive_dashboard.html</code>
                    </div>
                </li>
                <li>Dashboard opens in your browser!</li>
            </ol>
        </div>
        
        <div style="background: rgba(255,255,255,0.1); padding: 20px; border-radius: 8px; margin: 15px 0;">
            <h3 style="margin-top: 0; color: #FFD700;">Method 2: VS Code Live Server</h3>
            <ol style="line-height: 1.8; font-size: 15px;">
                <li>Install <strong>Live Server</strong> extension in VS Code</li>
                <li><strong>Open the folder</strong> containing your downloaded files</li>
                <li><strong>Right-click</strong> on <code>interactive_dashboard.html</code></li>
                <li><strong>Select "Open with Live Server"</strong></li>
                <li>Dashboard opens in your browser!</li>
            </ol>
        </div>
        
        
        <div style="background: rgba(255,255,0,0.15); padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #FFD700;">
            <h3 style="margin-top: 0; color: #FFD700;">⚠️ Important Notes:</h3>
            <ul style="line-height: 1.7; font-size: 14px; margin: 8px 0;">
                <li><strong>All files must be in the same folder:</strong>
                    <ul style="margin: 5px 0; font-size: 13px;">
                        <li><code>interactive_dashboard.html</code></li>
                        <li><code>all_points.geojson</code> (or chunked files)</li>
                        <li><code>chunks_manifest.json</code> (if large dataset)</li>
                    </ul>
                </li>
                <li><strong>Don't double-click the HTML file</strong> - it won't work due to CORS restrictions</li>
                <li><strong>Use a local server</strong> (any of the methods above) for full functionality</li>
                <li><strong>Keep the server running</strong> while viewing the dashboard</li>
            </ul>
        </div>
        
        <div style="text-align: center; margin-top: 20px; padding: 15px; background: rgba(255,255,255,0.1); border-radius: 8px;">
            <p style="font-size: 16px; margin: 0;">
                📁 Your files are in: <code style="background: rgba(0,0,0,0.3); padding: 6px 12px; border-radius: 4px; font-size: 14px;">{output_dir}</code>
            </p>
        </div>
    </div>
    '''
    
    display(HTML(instructions_html))
    print(f"📁 Download all files from: {output_dir}")

In [ ]:
def open_colab_dashboard_direct(output_dir):
    """Open interactive dashboard directly in Colab"""
    try:
        import http.server
        import socketserver
        import socket
        import threading
        import os

        print("🎯 Opening interactive dashboard...")

        # Find dashboard HTML file (CHANGED FROM interactive_map.html)
        dashboard_file = os.path.join(output_dir, 'interactive_dashboard.html')
        if not os.path.exists(dashboard_file):
            print("❌ Interactive dashboard file not found")
            return

        # Simple CORS server (same as before)
        class CORSHTTPRequestHandler(http.server.SimpleHTTPRequestHandler):
            def __init__(self, *args, **kwargs):
                if IN_COLAB and '/content/drive/MyDrive' in output_dir:
                    self.directory = '/content'
                elif IN_COLAB:
                    self.directory = '/content'
                else:
                    self.directory = '.'
                super().__init__(*args, directory=self.directory, **kwargs)

            def end_headers(self):
                self.send_header('Access-Control-Allow-Origin', '*')
                self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
                self.send_header('Access-Control-Allow-Headers', '*')
                super().end_headers()

        # Find free port
        def find_free_port():
            import random
            port_range = list(range(49152, 65536))
            random.shuffle(port_range)
            for port in port_range[:20]:
                try:
                    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                        s.bind(('', port))
                        return port
                except OSError:
                    continue
            return None

        port = find_free_port()
        if not port:
            print("❌ No available ports found")
            return

        # Start server
        httpd = socketserver.TCPServer(("", port), CORSHTTPRequestHandler)
        server_thread = threading.Thread(target=httpd.serve_forever)
        server_thread.daemon = True
        server_thread.start()

        print(f"✅ Server started on port {port}")

        # Calculate relative path
        if IN_COLAB and '/content/drive/MyDrive' in output_dir:
            relative_path = output_dir.replace('/content/', '')
        elif IN_COLAB:
            relative_path = output_dir.replace('/content/', '')
        else:
            relative_path = os.path.relpath(output_dir)

        # Try Colab port forwarding and open directly
        try:
            from google.colab import output
            from google.colab.output import eval_js

            output.serve_kernel_port_as_window(port)
            public_url = eval_js(f"google.colab.kernel.proxyPort({port})")
            # CHANGED: Use interactive_dashboard.html instead of interactive_map.html
            dashboard_url = f"{public_url}/{relative_path}/interactive_dashboard.html"

            # Open directly in new tab
            from IPython.display import display, Javascript
            display(Javascript(f'window.open("{dashboard_url}", "_blank");'))

            print(f"🌍 Interactive dashboard opened in new tab: {dashboard_url}")

        except Exception as e:
            print(f"⚠️ Port forwarding failed: {e}")
            # Fallback to local URL
            dashboard_url = f"http://localhost:{port}/{relative_path}/interactive_dashboard.html"

            # Try to open locally
            try:
                from IPython.display import display, Javascript
                display(Javascript(f'window.open("{dashboard_url}", "_blank");'))
                print(f"🔒 Dashboard opened locally: {dashboard_url}")
            except:
                print(f"🔗 Dashboard URL: {dashboard_url}")
                print("💡 Copy the URL above and paste it in a new browser tab")

    except Exception as e:
        print(f"❌ Failed to open interactive dashboard: {e}")
        # Final fallback - download
        try:
            from google.colab import files
            files.download(os.path.join(output_dir, 'interactive_dashboard.html'))
            print("📥 Downloaded interactive dashboard HTML file instead")
        except:
            print("❌ Could not open or download dashboard")


def start_local_dashboard_server(output_dir):
    """Start local server for interactive dashboard"""
    try:
        import http.server
        import socketserver
        import socket
        import threading
        import webbrowser
        import time

        print("🌐 Starting local server for interactive dashboard...")

        # Setup CORS-enabled server
        class CORSHTTPRequestHandler(http.server.SimpleHTTPRequestHandler):
            def __init__(self, *args, **kwargs):
                parent_dir = os.path.dirname(output_dir)
                if not parent_dir:
                    parent_dir = '.'
                super().__init__(*args, directory=parent_dir, **kwargs)

            def end_headers(self):
                self.send_header('Access-Control-Allow-Origin', '*')
                self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
                self.send_header('Access-Control-Allow-Headers', '*')
                super().end_headers()

        def find_free_port(start_port=8080, max_attempts=20):
            for port in range(start_port, start_port + max_attempts):
                try:
                    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                        s.bind(('', port))
                        return port
                except OSError:
                    continue
            raise OSError("No free ports available")

        # Find available port
        try:
            port = find_free_port()
            print(f"🔍 Found available port: {port}")
        except OSError as e:
            print(f"❌ No available ports found: {e}")
            return

        # Start local server
        httpd = socketserver.TCPServer(("", port), CORSHTTPRequestHandler)
        server_thread = threading.Thread(target=httpd.serve_forever)
        server_thread.daemon = True
        server_thread.start()
        print(f"✅ Local server started on port {port}")

        # Calculate the relative path to the dashboard
        relative_path = os.path.relpath(output_dir, os.path.dirname(output_dir))
        # CHANGED: Use interactive_dashboard.html instead of interactive_map.html
        dashboard_url = f"http://localhost:{port}/{relative_path}/interactive_dashboard.html"

        print(f"🗺️ Interactive dashboard URL: {dashboard_url}")
        print("🌐 Opening in your default browser...")

        # Small delay to ensure server is ready
        time.sleep(1)

        # Open in default browser
        webbrowser.open(dashboard_url)

        print("✅ Interactive dashboard opened!")
        print("💡 Server will keep running in the background")
        print("🔄 If the dashboard doesn't load, wait a moment and refresh the browser")

    except Exception as e:
        print(f"❌ Failed to start local server: {e}")
        print("📁 Fallback: Opening results folder...")
        open_local_folder(os.path.join(output_dir, 'interactive_dashboard.html'))


def show_simple_results_interface(results):
    """Simple results interface with dashboard button"""
    
    download_buttons = []
    
    # 1. Interactive Dashboard button
    if results['files'].get('dashboard'):
        if IN_COLAB:
            # Colab: Direct opening
            btn_dashboard = widgets.Button(
                description='Open Interactive Dashboard',
                button_style='primary',
                layout=widgets.Layout(width='100%', margin='5px 0')
            )
            btn_dashboard.on_click(lambda b: open_colab_dashboard_direct(results['output_dir']))
        elif IS_LOCAL:
            # Local: Start server
            btn_dashboard = widgets.Button(
                description='Open Interactive Dashboard',
                button_style='primary',
                layout=widgets.Layout(width='100%', margin='5px 0')
            )
            btn_dashboard.on_click(lambda b: start_local_dashboard_server(results['output_dir']))
        else:
            # Cloud environments (Kaggle, SageMaker, etc.): Show instructions
            with preview_image:
                show_hosting_instructions(results['output_dir'])
            # No button needed - instructions already shown
            pass
                
        if IN_COLAB or IS_LOCAL:
            download_buttons.append(btn_dashboard)
    
    
    # 2-4. Data download buttons - ONLY show for Colab and Local environments
    # Hidden for cloud environments where console output doesn't help
    if IN_COLAB or IS_LOCAL:
        if results['files']['frontier']:
            btn_geojson = widgets.Button(
                description='Download Frontier (GeoJSON)',
                button_style='',
                layout=widgets.Layout(width='100%', margin='2px 0')
            )
            if IN_COLAB:
                btn_geojson.on_click(lambda b: download_file_simple(results['files']['frontier']))
            elif IS_LOCAL:
                btn_geojson.on_click(lambda b: open_local_folder(results['files']['frontier']))
            else:
                # Cloud: Just show path for manual download
                btn_geojson.on_click(lambda b: print(f"📁 Download from: {results['files']['frontier']}"))
            download_buttons.append(btn_geojson)
        
        if results['files']['rank']:
            btn_rank = widgets.Button(
                description='Download Pareto Rank (GeoTIFF)',
                button_style='',
                layout=widgets.Layout(width='100%', margin='2px 0')
            )
            if IN_COLAB:
                btn_rank.on_click(lambda b: download_file_simple(results['files']['rank']))
            else:
                btn_rank.on_click(lambda b: open_local_folder(results['files']['rank']))
            download_buttons.append(btn_rank)
        
        if results['files']['percentiles']:
            btn_percentiles = widgets.Button(
                description='Download Pareto Rank Percentiles (GeoTIFF)',
                button_style='',
                layout=widgets.Layout(width='100%', margin='2px 0')
            )
            if IN_COLAB:
                btn_percentiles.on_click(lambda b: download_file_simple(results['files']['percentiles']))
            else:
                btn_percentiles.on_click(lambda b: open_local_folder(results['files']['percentiles']))
            download_buttons.append(btn_percentiles)
    else:
        # For cloud environments: Just skip the download buttons
        # Instructions already explain how to download files via platform's file browser
        pass
    
    return download_buttons

In [ ]:
# 🌍 DISPLAY THE INTERFACE - Exact replica of index.html layout

# Clear any previous results when cell is re-run
try:
    clear_all_results()
except:
    pass

# Header matching Canada.ca styling
header_html = '''
<div style="background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); color: white; padding: 20px; margin-bottom: 20px; border-radius: 8px;">
    <h1 style="margin: 0; font-size: 28px; font-weight: bold; text-align: center;">
        Canada critical mineral priority mapper 
    </h1>
    <p style="margin: 10px 0 0 0; text-align: center; font-size: 16px; opacity: 0.9;">
        Upload and process geospatial priority data for mineral assessments
    </p>
</div>
'''

# 🆕 CREATE AOI WIDGETS
aoi_nw_lat = widgets.FloatText(
    value=np.nan,  # Use NaN instead of None
    description='NW Latitude:',
    placeholder='e.g., 60.0',
    disabled=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='48%', margin='2px')
)

aoi_nw_lon = widgets.FloatText(
    value=np.nan,  # Use NaN instead of None
    description='NW Longitude:',
    placeholder='e.g., -120.0',
    disabled=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='48%', margin='2px')
)

aoi_se_lat = widgets.FloatText(
    value=np.nan,  # Use NaN instead of None
    description='SE Latitude:',
    placeholder='e.g., 50.0',
    disabled=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='48%', margin='2px')
)

aoi_se_lon = widgets.FloatText(
    value=np.nan,  # Use NaN instead of None
    description='SE Longitude:',
    placeholder='e.g., -110.0',
    disabled=False,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='48%', margin='2px')
)

aoi_clear_btn = widgets.Button(
    description='Clear AOI',
    button_style='',
    layout=widgets.Layout(width='100px', margin='5px 0')
)

def clear_aoi(button):
    """Clear AOI coordinates"""
    aoi_nw_lat.value = np.nan
    aoi_nw_lon.value = np.nan
    aoi_se_lat.value = np.nan
    aoi_se_lon.value = np.nan

aoi_clear_btn.on_click(clear_aoi)


def create_colab_map_preview_inline(output_dir):
    """Create externally accessible interactive map preview with public URL - INTEGRATED VERSION"""
    try:
        from IPython.display import display, HTML
        from pathlib import Path
        import json
        import os
        import http.server
        import socketserver
        import socket
        import threading

        print("🗺️ Creating externally accessible interactive map...")

        # Find HTML and GeoJSON files
        html_file = None
        geojson_file = None
        all_files = []

        for file_path in Path(output_dir).glob('*'):
            if file_path.is_file():
                all_files.append(file_path)
                if file_path.suffix == '.html':
                    html_file = file_path
                elif file_path.suffix == '.geojson':
                    geojson_file = file_path

        # Create file list for downloads
        files_info = []
        for file_path in all_files:
            if isinstance(file_path, str):
                file_path = Path(file_path)
            size = file_path.stat().st_size
            size_str = f"{size/1024/1024:.1f} MB" if size > 1024*1024 else f"{size/1024:.0f} KB"
            files_info.append({
                'name': file_path.name,
                'path': str(file_path),
                'size': size_str,
                'type': file_path.suffix
            })

        # 🌍 NEW: Create external accessible map server with correct path handling
        if html_file and geojson_file:
            try:
                # Setup CORS-enabled server that serves from the root directory structure
                class CORSHTTPRequestHandler(http.server.SimpleHTTPRequestHandler):
                    def __init__(self, *args, **kwargs):
                        # Set the directory to serve from the root that contains the full path
                        # This ensures paths like /drive/MyDrive/ESG_Mapper_Results/... work
                        if IN_COLAB and '/content/drive/MyDrive' in output_dir:
                            # For Google Drive mounted paths, serve from /content
                            self.directory = '/content'
                        elif IN_COLAB:
                            # For local Colab storage, serve from /content
                            self.directory = '/content'
                        else:
                            # For local development, serve from current directory
                            self.directory = '.'
                        super().__init__(*args, directory=self.directory, **kwargs)

                    def end_headers(self):
                        self.send_header('Access-Control-Allow-Origin', '*')
                        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
                        self.send_header('Access-Control-Allow-Headers', '*')
                        super().end_headers()

                def find_free_port(start_port=49152, max_attempts=20):
                    """Find an available port starting from start_port"""
                    import random
                    # Use Windows dynamic port range to avoid permission issues
                    port_range = list(range(49152, 65536))

                    random.shuffle(port_range)
                    for port in port_range[:max_attempts]:
                        try:
                            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                                s.bind(('', port))
                                return port
                        except OSError:
                            continue
                    raise OSError("No free ports available")

                # Find available port
                try:
                    port = find_free_port()
                    print(f"🔍 Found available port: {port}")
                except OSError as e:
                    print(f"❌ No available ports found: {e}")
                    print("💡 Try restarting the runtime to free up ports")
                    return

                # Start local server (don't change directory - serve from root)
                # original_dir = os.getcwd()

                # Create server that can handle the full path structure
                httpd = socketserver.TCPServer(("", port), CORSHTTPRequestHandler)
                server_thread = threading.Thread(target=httpd.serve_forever)
                server_thread.daemon = True
                server_thread.start()
                print(f"✅ Local server started on port {port}")

                # Calculate the relative path from server root to the output directory
                if IN_COLAB and '/content/drive/MyDrive' in output_dir:
                    # For Google Drive paths, the URL path should be relative to /content
                    relative_path = output_dir.replace('/content/', '')
                elif IN_COLAB:
                    # For local Colab storage
                    relative_path = output_dir.replace('/content/', '')
                else:
                    # For local development
                    relative_path = os.path.relpath(output_dir)

                # Try Colab's built-in port forwarding (BEST METHOD)
                public_url = None
                try:
                    from google.colab import output
                    from google.colab.output import eval_js

                    # Enable port forwarding
                    output.serve_kernel_port_as_window(port)

                    # Get public URL
                    public_url = eval_js(f"google.colab.kernel.proxyPort({port})")

                    # Construct the full URL to the interactive map
                    map_url = f"{public_url}/{relative_path}/interactive_map.html"

                    # Display the external map interface
                    display(HTML(f'''
                    <div style="background: linear-gradient(135deg, #4CAF50 0%, #45a049 100%); color: white; padding: 30px; margin: 25px 0; border-radius: 15px; box-shadow: 0 10px 30px rgba(76,175,80,0.4);">
                        <h2 style="margin-top: 0; text-align: center; font-size: 24px;">🌍 External Interactive Map Server</h2>
                        <p style="text-align: center; font-size: 16px; opacity: 0.95; margin-bottom: 25px;">Your analysis results are now accessible from any browser, any device!</p>

                        <div style="background: rgba(255,255,255,0.1); padding: 25px; border-radius: 12px; margin: 20px 0;">
                            <div style="text-align: center; margin: 20px 0;">
                                <h3 style="margin: 10px 0; font-size: 20px;">
                                    <a href="{map_url}" target="_blank"
                                       style="color: #FFD700; text-decoration: none; background: rgba(255,215,0,0.2); padding: 15px 25px; border-radius: 8px; display: inline-block; transition: all 0.3s;">
                                        🗺️ Open Interactive Map
                                    </a>
                                </h3>
                                <p style="margin: 15px 0; font-size: 14px; opacity: 0.9;">
                                    Map URL: <code style="background: rgba(0,0,0,0.2); padding: 5px 10px; border-radius: 4px; font-size: 12px; word-break: break-all;">{map_url}</code>
                                </p>
                                <a href={map_url}>View interactive map</a>
                                <p style="margin: 15px 0; font-size: 14px; opacity: 0.9;">
                                    Server Root: <code style="background: rgba(0,0,0,0.2); padding: 5px 10px; border-radius: 4px; font-size: 12px;">{public_url}</code>
                                </p>
                            </div>

                            <div style="text-align: center; margin-top: 15px; padding: 15px; background: rgba(255,255,255,0.1); border-radius: 8px;">
                                <p style="margin: 5px 0; font-weight: bold; font-size: 14px;">🌐 External Access Features:</p>
                                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin: 10px 0; font-size: 13px;">
                                    <span>• ✅ Accessible from any browser</span>
                                    <span>• 📱 Works on phones & tablets</span>
                                    <span>• 🔗 Share URL with anyone</span>
                                    <span>• 🌍 No Colab session required</span>
                                </div>
                                <p style="margin: 10px 0 5px 0; font-size: 12px; opacity: 0.8;">
                                    💡 Map will remain accessible as long as this Colab session is running
                                </p>
                            </div>

                            <div style="text-align: center; margin-top: 15px; padding: 15px; background: rgba(0,0,0,0.1); border-radius: 8px;">
                                <p style="margin: 5px 0; font-weight: bold; font-size: 12px;">🔍 Debug Info:</p>
                                <p style="margin: 5px 0; font-size: 11px; opacity: 0.8;">Output Dir: {output_dir}</p>
                                <p style="margin: 5px 0; font-size: 11px; opacity: 0.8;">Relative Path: {relative_path}</p>
                                <p style="margin: 5px 0; font-size: 11px; opacity: 0.8;">Server Directory: {'/content' if IN_COLAB else '.'}</p>
                            </div>
                        </div>
                    </div>
                    '''))

                    print(f"🌍 SUCCESS! Map accessible globally at: {map_url}")
                    print(f"📂 Server root: {public_url}")
                    print(f"🗂️ Serving from: {'/content' if IN_COLAB else '.'}")
                    print(f"📍 Relative path: {relative_path}")

                except Exception as colab_error:
                    print(f"⚠️ Colab port forwarding failed: {colab_error}")


                     # Final fallback: Show local server info with correct path
                    map_url = f"http://localhost:{port}/{relative_path}/interactive_map.html"
                    display(HTML(f'''
                    <div style="background: #FFA726; color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
                        <h3>🔒 Local Server Started</h3>
                        <p>Server running at: <code>http://localhost:{port}</code></p>
                        <p>Map URL: <code>{map_url}</code></p>
                        <a href={map_url}>View interactive map</a>
                        <p>⚠️ Only accessible within this session</p>
                    </div>
                    '''))

                print("✅ External map server created successfully!")

            except Exception as server_error:
                print(f"⚠️ Server creation failed: {server_error}")
                print("📁 Showing download options instead...")
        else:
            print("ℹ️ Interactive map files not found - showing download options")

        # Always show download interface (same as before)
        if files_info:
            display(HTML(f'''
            <div style="background: #e3f2fd; padding: 25px; margin: 25px 0; border-radius: 12px; border-left: 5px solid #2196F3;">
                <h2 style="color: #1565c0; margin-top: 0;">📥 Download Your Results</h2>
                <p style="color: #333; margin-bottom: 20px;">Download files for offline use or further analysis:</p>

                <div style="display: grid; gap: 12px; margin: 20px 0;">
            '''))

            for file_info in files_info:
                file_type_icons = {
                    '.tif': '📊', '.geojson': '📄', '.html': '🗺️', '.png': '🖼️', '.json': '📋'
                }
                file_descriptions = {
                    '.tif': 'GeoTIFF Raster (for GIS software)',
                    '.geojson': 'Vector Data (web mapping)',
                    '.html': 'Interactive Map (offline viewing)',
                    '.png': 'Static Image (reports)',
                    '.json': 'Analysis Data (structured)'
                }

                icon = file_type_icons.get(file_info['type'], '📁')
                desc = file_descriptions.get(file_info['type'], 'Data File')

                # Special styling for different file types
                if file_info['type'] == '.html':
                    gradient = "linear-gradient(135deg, #FF6B35 0%, #F7931E 100%)"
                    shadow_color = "rgba(255,107,53,0.3)"
                elif file_info['type'] == '.geojson':
                    gradient = "linear-gradient(135deg, #4CAF50 0%, #45a049 100%)"
                    shadow_color = "rgba(76,175,80,0.3)"
                else:
                    gradient = "linear-gradient(135deg, #2196F3 0%, #21CBF3 100%)"
                    shadow_color = "rgba(33,150,243,0.3)"

                display(HTML(f'''
                <div style="padding: 18px; background: white; border-radius: 10px; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 3px 12px rgba(0,0,0,0.1); transition: all 0.2s; border: 1px solid #e0e0e0; margin: 10px 0;">
                    <div style="display: flex; align-items: center; gap: 15px;">
                        <span style="font-size: 26px;">{icon}</span>
                        <div>
                            <div style="font-weight: bold; color: #333; font-size: 15px;">{file_info['name']}</div>
                            <div style="color: #666; font-size: 13px; margin: 3px 0;">{desc}</div>
                            <div style="color: #888; font-size: 12px;">{file_info['size']}</div>
                        </div>
                    </div>
                    <button onclick="downloadFile('{file_info['path']}')" style="
                        background: {gradient};
                        color: white; border: none; padding: 12px 20px; border-radius: 8px;
                        cursor: pointer; font-weight: bold; transition: all 0.3s;
                        box-shadow: 0 4px 15px {shadow_color};
                        font-size: 13px;
                    " onmouseover="this.style.transform='scale(1.05)'"
                       onmouseout="this.style.transform='scale(1)'">📥 Download</button>
                </div>
                '''))

            # Add bulk download button (same JavaScript as before)
            display(HTML(f'''
                </div>

                <div style="text-align: center; margin: 25px 0 15px 0; padding: 20px; background: linear-gradient(135deg, #9C27B0 0%, #E91E63 100%); border-radius: 12px; box-shadow: 0 6px 20px rgba(156,39,176,0.3);">
                    <button onclick="downloadAllFiles()" style="
                        background: rgba(255,255,255,0.2); color: white; border: 2px solid white;
                        padding: 15px 30px; border-radius: 10px; cursor: pointer; font-weight: bold;
                        font-size: 16px; transition: all 0.3s;
                    " onmouseover="this.style.background='rgba(255,255,255,0.3)'"
                       onmouseout="this.style.background='rgba(255,255,255,0.2)'">
                       📦 Download All Files
                    </button>
                    <p style="color: white; margin: 12px 0 0 0; opacity: 0.9; font-size: 14px;">
                        Downloads all {len(files_info)} files automatically
                    </p>
                </div>

                <div style="background: #f0f8ff; padding: 18px; border-radius: 8px; margin-top: 20px;">
                    <h4 style="color: #1565c0; margin-top: 0;">💡 Usage Tips:</h4>
                    <ul style="color: #333; line-height: 1.6; margin: 10px 0; font-size: 14px;">
                        <li><strong>External Map Access:</strong> Use the public URL above to view from any device</li>
                        <li><strong>Offline Map:</strong> Download HTML + GeoJSON, place in same folder</li>
                        <li><strong>GIS Analysis:</strong> Import GeoTIFF files into QGIS, ArcGIS, etc.</li>
                        <li><strong>Web Sharing:</strong> Share the public URL or upload GeoJSON to web platforms</li>
                    </ul>
                </div>
            </div>

            <script>
            function downloadFile(filePath) {{
                try {{
                    console.log('Downloading:', filePath);
                    google.colab.files.download(filePath);
                }} catch (error) {{
                    console.error('Download failed:', error);
                    alert('Download failed: ' + error.message);
                }}
            }}

            function downloadAllFiles() {{
                const filePaths = [{', '.join([f"'{file_info['path']}'" for file_info in files_info])}];
                let downloaded = 0;

                console.log('Starting bulk download of', filePaths.length, 'files...');

                const downloadNext = () => {{
                    if (downloaded < filePaths.length) {{
                        try {{
                            console.log(`Downloading ${{downloaded + 1}}/${{filePaths.length}}:`, filePaths[downloaded]);
                            google.colab.files.download(filePaths[downloaded]);
                            downloaded++;
                            setTimeout(downloadNext, 1000);
                        }} catch (error) {{
                            console.error('Failed to download:', filePaths[downloaded], error);
                            downloaded++;
                            setTimeout(downloadNext, 500);
                        }}
                    }} else {{
                        console.log('All downloads completed!');
                        alert('✅ All files downloaded successfully!');
                    }}
                }};

                downloadNext();
            }}
            </script>
            '''))

    except Exception as e:
        print(f"❌ External map server creation failed: {e}")
        import traceback
        traceback.print_exc()
        print("📁 Fallback: Files available for download manually")




# 📋 UPDATED DOWNLOAD BUTTON HANDLERS
def create_download_handlers(results):
    """Create download button handlers optimized for Colab"""

    def download_frontier(btn):
        if IN_COLAB:
            download_file_colab(results['files']['frontier'], 'pareto_frontier.geojson')
        else:
            print(f"📁 Frontier file: {results['files']['frontier']}")

    def download_rank(btn):
        if IN_COLAB:
            download_file_colab(results['files']['rank'], 'pareto_rank.tif')
        else:
            print(f"📁 Rank file: {results['files']['rank']}")

    def download_percentiles(btn):
        if IN_COLAB:
            download_file_colab(results['files']['percentiles'], 'pareto_percentiles.tif')
        else:
            print(f"📁 Percentiles file: {results['files']['percentiles']}")

    def view_interactive_map(btn):
        if IN_COLAB:
            download_file_colab(results['files']['interactive_map'], 'interactive_map.html')
            print("💡 Download the HTML file and open it in your browser")
        else:
            print(f"📁 Interactive map: {results['files']['interactive_map']}")

    def show_file_browser(btn):
        print("📁 Creating file browser...")
        serve_files_colab_simple(results['output_dir'])

    def start_local_server(btn):
        print("🌐 Starting local server...")
        serve_files_local_server(results['output_dir'])

    def download_all_files(btn):
        """Download all result files at once"""
        if IN_COLAB:
            print("📦 Downloading all result files...")
            for file_type, file_path in results['files'].items():
                if file_path and os.path.exists(file_path):
                    filename_map = {
                        'frontier': 'pareto_frontier.geojson',
                        'rank': 'pareto_rank.tif',
                        'percentiles': 'pareto_percentiles.tif',
                        'interactive_map': 'interactive_map.html',
                        'plot': 'result_map.png'
                    }
                    display_name = filename_map.get(file_type, os.path.basename(file_path))
                    download_file_colab(file_path, display_name)
                    time.sleep(0.5)  # Small delay between downloads
            print("✅ All files downloaded!")
        else:
            print(f"📁 All files available in: {results['output_dir']}")

    return {
        'download_frontier': download_frontier,
        'download_rank': download_rank,
        'download_percentiles': download_percentiles,
        'view_interactive_map': view_interactive_map,
        'show_file_browser': show_file_browser,
        'start_local_server': start_local_server,
        'download_all_files': download_all_files
    }

# 📥 COLAB DOWNLOAD FUNCTIONS (Updated)
def download_file_colab(file_path, display_name=None):
    """Download file using Colab's files.download()"""
    if IN_COLAB and os.path.exists(file_path):
        try:
            from google.colab import files
            files.download(file_path)
            print(f"✅ Downloaded: {display_name or os.path.basename(file_path)}")
        except Exception as e:
            print(f"❌ Download failed: {e}")
    else:
        print(f"❌ File not found or not in Colab: {file_path}")



left_column = widgets.VBox([
    widgets.HTML('<h2 style="color: #335075; margin-bottom: 10px;">Select Mineral Potential</h2>'),
    mineral_select,

    widgets.HTML('<h2 style="color: #335075; margin: 20px 0 10px 0;">Other Priorities</h2>'),
    infrastructure_check,
    protected_areas_check,
    species_risk_check,

    # NEW: Area of Interest (AOI) widgets
    widgets.HTML('<h2 style="color: #335075; margin: 20px 0 10px 0;">Area of Interest (Optional)</h2>'),
    widgets.HTML('<p style="margin: 5px 0; color: #666;">Enter bounding box coordinates to limit analysis area</p>'),

    widgets.HBox([aoi_nw_lat, aoi_nw_lon], layout=widgets.Layout(width='100%')),
    widgets.HBox([aoi_se_lat, aoi_se_lon], layout=widgets.Layout(width='100%')),
    aoi_clear_btn,


    # MAXIMIZE section
    widgets.HTML('<h2 style="color: #335075; margin: 20px 0 10px 0;">Custom Priority Files (Maximize)</h2>'),
    # widgets.HTML('<p style="margin: 5px 0; color: #4CAF50;">📈 Higher values = better (e.g., mineral prospectivity)</p>'),
    file_chooser,
    widgets.HBox([clear_btn]),
    file_display,

    # 🆕 NEW: MINIMIZE section
    widgets.HTML('<h2 style="color: #335075; margin: 20px 0 10px 0;">Custom Priority Files (Minimize)</h2>'),
    # widgets.HTML('<p style="margin: 5px 0; color: #f44336;">📉 Lower values = better (e.g., costs, distances)</p>'),
    file_chooser_minimize,
    widgets.HBox([clear_btn_minimize]),
    file_display_minimize,

    widgets.HTML('<div style="margin: 20px 0;"></div>'),
    process_btn,
    progress_bar,
    step_counter,
    status_output

], layout=widgets.Layout(width='45%', padding='10px'))

# Right column - Results (matching index.html structure)
right_column = widgets.VBox([
    widgets.HTML('<h2 style="color: #335075; margin-bottom: 10px;">Results</h2>'),
    preview_image,
    result_buttons

], layout=widgets.Layout(width='55%', padding='10px'))

# Main container
main_container = widgets.HBox([left_column, right_column])

# Display the complete interface
display(widgets.VBox([
    widgets.HTML(header_html),
    main_container
]))


print(f"\n🎉 Canada critical mineral priority mapper ready!")
print(f"📂 Results will be saved to: {WORK_DIR}")
